# DR-VERGE — Complete Pipeline Notebook (rev3)

**View-Evidence Relational Grading Engine** — dual-view diabetic retinopathy grading via
Complementarity-Shift Distillation (CSD). GEMASTIK XIX, Bidang VII KTI.

This single notebook runs the **entire experiment**, top to bottom, on a Colab GPU runtime:

1. Setup (Drive mount, dependencies, GPU check)
2. Dataset verification (Gate 1) — DRTiD primary, patient-wise official split
3. All source code inline: datasets, models (CORALHead / Teacher / Student), losses (CORAL, KD, CSD)
4. Smoke test
5. Backbone pretraining on APTOS (ResNet-50 for teacher, lightweight for student)
6. Teacher training (Gate 2)
7. Student baselines: macula-only, disc-only, no-distill, standard KD (3 seeds each for the core 3)
8. CSD signal check (Gate 3), grid search, final DR-VERGE training (3 seeds)
9. PTQ INT8 quantization (Gate 5)
10. Full evaluation, seed aggregation, clustered bootstrap CIs
11. Chart generation (all figures saved as PNG)
12. Final results dashboard

**Every checkpoint, metric CSV, and figure is saved to Google Drive** as it's produced, so a
disconnected Colab runtime never loses completed work — re-running a cell whose output already
exists on Drive skips straight to the next step where practical.

Source of truth for the method: `docs/overview.md`, `docs/[USED THIS] Technical Documentation.pdf`
(v2), `docs/judge.md` (external audit — fixes for its flags are implemented directly in this
notebook's model/loss code, not just described). Day-by-day plan: `docs/roadmap.md`.

---

## What changed in rev3 (and why)

rev2 completed successfully but returned a **null result at Gate 4** (`dual_csd` 0.406 vs
`dual_no_distill` 0.447, bootstrap CIs spanning zero). Reading rev2's own saved outputs showed the
null was driven by measurable pipeline defects rather than by the CSD idea itself. rev3 fixes those.
Full analysis in `experiment/results.md`; per-change rationale in `experiment/revision-notes-rev3.md`.

| # | Fix | Evidence from the rev2 run |
|---|---|---|
| 1 | **CORAL thresholds initialized from the label distribution** | rev2 started all 4 thresholds within 0.15 logits of each other (needed span: 3.24). Every condition, teacher included, predicted essentially only Grade 0 and Grade 4 — Grade 1/2/3 sensitivity was 0.00–0.04. |
| 2 | **`pos_weight` mode `sqrt`** | Raw inverse frequency gave 24.8× weight to threshold 3 (31 of 800 eyes are Grade 4), driving that same collapse and hurting QWK's distance-weighted score. |
| 3 | **Student capacity 34K → ~330K params** | rev2's conv stack was only 8,176 params (fusion MLP was 75% of the model), ~40× below the technical doc's own 0.3–0.4M target. All dual conditions landed within 0.04 QWK — a capacity ceiling, which makes RQ1 untestable. |
| 4 | **Scale-normalized CSD loss (`smoothl1_norm`)** | rev2 logged `L_CSD≈0.014` vs `L_task≈0.82`: under 0.5% of total loss. Measured here: gradient norm rises 1.17 → 44.96 (38×). |
| 5 | **Grid searches a meaningful β range** | rev2's grid only varied β over {0.5, 0.7}, all of which produced a negligible CSD contribution — it could never have found a working setting. |
| 6 | **Per-component gradient-norm logging** (judge.md Flag 6) | rev2 logged loss *values* only, so "CSD contributes nothing" stayed invisible until the logs were re-read by hand after the entire run finished. |
| 7 | **Shift-fidelity metrics** (judge.md Flag 10) | rev2 had no way to tell whether CSD transferred the shift pattern *independently* of whether QWK improved. |
| 8 | **Feature-KD control baseline** (judge.md Section G) | rev2 had no control isolating "decision-shift knowledge" from ordinary representation transfer. |
| 9 | **External dual-view gain** (judge.md Flag 8) | rev2 reported only internal gain, which judge.md explicitly says must not be conflated with the external one. |
| 10 | **Calibration: ECE + Brier** (judge.md Flag 4) | CSD is framed as distilling a *confidence* shift, so calibration is load-bearing — rev2 measured neither. |
| 11 | **TorchScript fix in `InteractionFusion`** | rev2's Gate 5 hit `Module 'InteractionFusion' has no attribute 'norm'` and fell back to a state_dict save instead of a real deployment artifact. |
| 12 | **INT8 dual-view gain measured** | rev2 left it `NaN`, so RQ2's actual sub-question ("does PTQ preserve the dual-view advantage?") was never answered. |
| 13 | **AdamW + cosine LR, 40 epochs, patience 8** | rev2's val QWK oscillated hard (0.51→0.35→0.46 between epochs), wasting the patience budget and making "best epoch" partly luck. |

**Honest note:** these fixes remove the defects that made rev2's RQ1 test *uninformative*. They do
not guarantee CSD will win — that remains an empirical question this notebook is designed to answer
truthfully either way. What rev3 guarantees is that whatever Gate 4 reports is a real result about
the method, not an artifact of a collapsed ordinal head, a capacity-capped student, or a loss term
with no gradient.

---

**Before running:** set `DRIVE_BASE` in the Config cell to a folder in your Google Drive, and make
sure `dataset/DRTiD/DRTiD/...` and `dataset/APTOS/...` exist there (or use the upload cell to get
them there) — see the Setup section below for both paths.

**Re-running after rev2:** every checkpoint from rev2 is architecturally incompatible with rev3
(different CORAL parameterization, different backbone widths). `checkpoint_is_compatible()` detects
this automatically and retrains rather than loading stale weights — including the two APTOS
backbones, which *do* need re-pretraining this time because the student width changed.

## 1. Setup

In [ ]:
# 1.1 GPU check -- stop here if this doesn't show a GPU (Runtime > Change runtime type > GPU)
!nvidia-smi

In [ ]:
# 1.2 Mount Google Drive -- this is where the dataset lives and where all outputs get saved,
# so results and checkpoints survive a disconnected runtime.
from google.colab import drive
drive.mount('/content/drive')

### 1.3 Get the dataset onto Drive (one-time, skip if already there)

Two options:

- **Already synced**: if `dataset/DRTiD` and `dataset/APTOS` already exist under `DRIVE_BASE`
  (set below), skip straight to 1.4.
- **First time**: zip your local `dataset/` folder (DRTiD + APTOS subfolders) and upload it via
  the cell below, or upload directly into Drive through the Drive web UI (faster for large
  folders than a browser upload) and skip this cell.

In [ ]:
# 1.3 (optional) Upload a dataset.zip via the browser if it is not already on Drive.
# Skip this cell if dataset/DRTiD and dataset/APTOS already exist under DRIVE_BASE.
RUN_UPLOAD_CELL = False  # flip to True if you need to upload

if RUN_UPLOAD_CELL:
    from google.colab import files
    import zipfile, os
    uploaded = files.upload()  # select dataset.zip (contains DRTiD/ and APTOS/ at top level)
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall("/content/drive/MyDrive/DR-VERGE/dataset")
    print("Extracted to /content/drive/MyDrive/DR-VERGE/dataset")

In [ ]:
# 1.4 Install dependencies. Colab already ships a CUDA-enabled torch -- we deliberately do NOT
# reinstall torch/torchvision here to avoid clobbering the platform's GPU build with a
# mismatched one (this is exactly the trap that cost time locally where the dev machine had a
# CPU-only torch and an old driver -- see docs/roadmap.md Day 1 notes).
!pip install -q albumentations scikit-learn==1.9.0 pandas tqdm pyyaml

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU detected -- check Runtime > Change runtime type > GPU."

## 2. Global Config

In [ ]:
import os

# ---- EDIT THIS to your Drive folder ----
DRIVE_BASE = "/content/drive/MyDrive/DR-VERGE"

DATASET_ROOT   = f"{DRIVE_BASE}/dataset"
# DRTiD ships nested as DRTiD/DRTiD/... on the original distribution, but folders sometimes get
# flattened to DRTiD/... during manual reorganization -- auto-detect whichever is actually
# present rather than hardcoding one and silently breaking if your Drive layout differs from
# whatever a previous run happened to use.
def _resolve_drtid_root(dataset_root):
    nested = f"{dataset_root}/DRTiD/DRTiD"
    flat = f"{dataset_root}/DRTiD"
    if os.path.exists(f"{nested}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
        return nested
    if os.path.exists(f"{flat}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
        return flat
    return nested  # neither exists yet -- fall through to the Gate 1 check below for a clear error

DRTID_ROOT     = _resolve_drtid_root(DATASET_ROOT)
APTOS_ROOT     = f"{DATASET_ROOT}/APTOS"
SPLITS_DIR     = f"{DRIVE_BASE}/splits"
CKPT_DIR       = f"{DRIVE_BASE}/checkpoints"
RESULTS_DIR    = f"{DRIVE_BASE}/results"
FIGURES_DIR    = f"{RESULTS_DIR}/figures"
METRICS_DIR    = f"{RESULTS_DIR}/metrics"
LOGS_DIR       = f"{RESULTS_DIR}/logs"

for d in [SPLITS_DIR, CKPT_DIR, f"{CKPT_DIR}/pretrained_backbones", f"{CKPT_DIR}/teacher",
          f"{CKPT_DIR}/student", RESULTS_DIR, FIGURES_DIR, METRICS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Sanity check dataset is where we expect it
_expected = [
    f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv",
    f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv",
    f"{DRTID_ROOT}/Original Images",
    f"{APTOS_ROOT}/train_1.csv",
    f"{APTOS_ROOT}/valid.csv",
    f"{APTOS_ROOT}/train_images/train_images",
    f"{APTOS_ROOT}/val_images/val_images",
]
_missing = [p for p in _expected if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError(
        "Dataset not found at the expected Drive path -- see Section 1.3:\n" + "\n".join(_missing)
    )
print(f"Dataset check OK. DRTID_ROOT resolved to: {DRTID_ROOT}")

SEEDS = [42, 123, 2026]
PRIMARY_SEED = 42
IMG_SIZE = 224
NUM_CLASSES = 5
NUM_THRESHOLDS = NUM_CLASSES - 1

# ---- REV3 knobs (see experiment/revision-notes-rev3.md for the evidence behind each) ----
# "sqrt" tames the degenerate raw inverse-frequency weight (24.8x at threshold 3 on DRTiD) that
# collapsed v2's predictions onto Grades 0 and 4 only. "full" reproduces v2 behavior, "none"
# disables imbalance weighting entirely.
POS_WEIGHT_MODE = "sqrt"
# Student backbone width. v2 shipped a backbone with only ~8.2K parameters in the conv stack
# (the fusion MLP was 75% of the whole 34K model) -- roughly 40x below the technical doc's own
# 0.3-0.4M target, which capacity-capped every dual-view condition at the same QWK regardless of
# distillation method. This channel plan lands the whole student near ~330K.
STUDENT_CHANNELS = (32, 64, 96, 128, 160, 192, 224)

## 3. Reproducibility utils

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_generator(seed: int) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

def compute_pos_weights(train_csv: str, num_thresholds: int = NUM_THRESHOLDS, grade_col: str = "grade",
                         mode: str = POS_WEIGHT_MODE) -> torch.Tensor:
    # pos_weight_k = N_negative_k / N_positive_k, for F.binary_cross_entropy_with_logits.
    # Computed ONCE from the training split, reused for every condition/seed so that
    # differences in performance come from the distillation method, not from some
    # conditions accidentally getting better imbalance handling than others.
    #
    # REV3 CHANGE -- mode. The v2 run used the raw ratio ("full"), which on DRTiD gives
    # pos_weight=24.8 at threshold k=3 (only 31 of 800 training eyes are Grade 4). That extreme
    # weight pushed P(y>3) up so aggressively that the v2 models predicted almost exclusively
    # Grade 0 or Grade 4: measured per-grade sensitivity was Grade0 0.85-0.94 and Grade4
    # 0.95-1.00, but Grade1 0.00-0.04, Grade2 0.01-0.03, Grade3 0.00-0.03 -- i.e. the middle
    # three grades were effectively never predicted, by every condition INCLUDING the teacher.
    # QWK punishes that hard (quadratic distance penalty), which capped every condition near
    # 0.40-0.45 and left almost no headroom for any distillation method to show a difference.
    # "sqrt" keeps the direction of the imbalance correction but tames its magnitude
    # (24.8 -> 5.0 at k=3), which is the standard compromise when a raw inverse-frequency weight
    # is degenerate. Also directly relevant to judge.md Flag 4: a milder weight distorts the
    # sigmoid outputs' probability interpretation less, which matters because CSD's Delta is
    # defined on exactly those outputs.
    df = pd.read_csv(train_csv)
    grades = df[grade_col].values
    weights = []
    for k in range(num_thresholds):
        pos = int((grades > k).sum())
        neg = int((grades <= k).sum())
        if pos == 0 or neg == 0:
            raise ValueError(f"pos_weight threshold k={k} has pos={pos}, neg={neg} -- degenerate split.")
        ratio = neg / pos
        if mode == "full":
            weights.append(ratio)
        elif mode == "sqrt":
            weights.append(math.sqrt(ratio))
        elif mode == "none":
            weights.append(1.0)
        else:
            raise ValueError(f"unknown pos_weight mode: {mode}")
    return torch.tensor(weights, dtype=torch.float32)


def compute_init_thresholds(train_csv: str, num_thresholds: int = NUM_THRESHOLDS,
                             grade_col: str = "grade", eps: float = 1e-3):
    """Logit of the empirical cumulative label distribution: logit(P(y>k)) for k=0..K-2.

    REV3 ADDITION. Used to initialize CORALHead's ordered thresholds at the dataset's actual
    marginal distribution instead of at an arbitrary near-collapsed point. See CORALHead's
    docstring for why this matters -- it is the single highest-impact fix in rev3.
    """
    df = pd.read_csv(train_csv)
    grades = df[grade_col].values
    thresholds = []
    for k in range(num_thresholds):
        p = float((grades > k).mean())
        p = min(max(p, eps), 1.0 - eps)
        thresholds.append(math.log(p / (1.0 - p)))
    return thresholds

def robust_torch_load(path, map_location=None, retries=6, initial_delay=1.0):
    # Google Drive's FUSE mount has a short, well-documented sync lag: a file written in one line
    # can raise FileNotFoundError on a plain torch.load() of that exact same path on the very next
    # line, in the same running process. Observed directly on a real run: a teacher checkpoint
    # saved at the end of the freeze phase failed to load moments later when starting the finetune
    # phase. Retries with exponential backoff rather than letting a filesystem race kill a training
    # run that otherwise worked correctly. Use this in place of plain torch.load(...) for every
    # checkpoint this notebook itself may have just written, not just ones from a prior session.
    import time as _time
    delay = initial_delay
    last_err = None
    for attempt in range(retries):
        try:
            return torch.load(path, map_location=map_location)
        except (FileNotFoundError, OSError) as e:
            last_err = e
            if attempt < retries - 1:
                print(f"robust_torch_load: '{path}' not readable yet (attempt {attempt+1}/{retries}), retrying in {delay:.1f}s...")
                _time.sleep(delay)
                delay *= 1.5
    raise last_err

def robust_torch_save(obj, path, retries=6, initial_delay=1.0):
    # Mirrors robust_torch_load, for the write side. Google Drive's FUSE mount can lag behind its
    # own directory creation too, not just file writes: confirmed directly on a real run --
    # os.makedirs(parent, exist_ok=True) returned successfully, but torch.save()'s lower-level
    # file writer then raised "RuntimeError: Parent directory ... does not exist" on the very
    # first student checkpoint save moments later. Retries the makedirs+save pair together rather
    # than letting a transient Drive sync lag kill a training run. Use this in place of plain
    # torch.save(...) for every checkpoint this notebook writes.
    import time as _time
    delay = initial_delay
    last_err = None
    parent = os.path.dirname(path)
    for attempt in range(retries):
        try:
            if parent:
                os.makedirs(parent, exist_ok=True)
            torch.save(obj, path)
            return
        except (RuntimeError, OSError, FileNotFoundError) as e:
            last_err = e
            if attempt < retries - 1:
                print(f"robust_torch_save: saving to '{path}' failed (attempt {attempt+1}/{retries}): {e} -- retrying in {delay:.1f}s...")
                _time.sleep(delay)
                delay *= 1.5
    raise last_err

def checkpoint_is_compatible(ckpt_path, model, unwrap_key="model_state"):
    # Returns True only if ckpt_path exists AND its saved state's keys+shapes exactly match
    # `model`'s CURRENT architecture. Guards the "skip training if checkpoint already exists"
    # pattern used throughout this notebook: without this, changing a model's architecture (e.g.
    # the InteractionFusion redesign in Section 6) and then re-running against an old checkpoint
    # on Drive would either silently reuse stale/incompatible weights or crash confusingly deep
    # inside a later cell. If incompatible, prints why and returns False so the caller retrains.
    #
    # Deliberately does NOT call model.load_state_dict(..., strict=True) on the live model --
    # PyTorch's strict load can copy some matching-name parameters into the module in place
    # before it finally raises on a key/shape mismatch elsewhere, which would leave `model`
    # partially mutated (some fresh-init, some stale-loaded) if the caller then falls through to
    # training on it. Comparing key sets and shapes directly against a state_dict() snapshot
    # never touches the real model, so a "not compatible" verdict is always side-effect-free.
    if not os.path.exists(ckpt_path):
        return False
    try:
        raw = robust_torch_load(ckpt_path, map_location="cpu")
        state = raw[unwrap_key] if (unwrap_key and isinstance(raw, dict) and unwrap_key in raw) else raw
        current = model.state_dict()
        if set(state.keys()) != set(current.keys()):
            missing = set(current.keys()) - set(state.keys())
            unexpected = set(state.keys()) - set(current.keys())
            raise RuntimeError(f"key mismatch (missing={list(missing)[:5]}, unexpected={list(unexpected)[:5]})")
        for k in state:
            if state[k].shape != current[k].shape:
                raise RuntimeError(f"shape mismatch at '{k}': checkpoint={tuple(state[k].shape)} vs model={tuple(current[k].shape)}")
        return True
    except Exception as e:
        print(f"{ckpt_path} exists but does NOT match the current model architecture "
              f"({type(e).__name__}: {str(e)[:300]}) -- retraining instead of skipping.")
        return False

## 4. Gate 1 — Dataset split (DRTiD)

DRTiD ships an **official** train/test split (`a. DR_grade_Training.csv` / `b. DR_grade_Testing.csv`)
— used as-is, never re-shuffled. We only carve our own train/val split out of the 1000 official
training rows, since DRTiD gives no val set. `_1` = Macula, `_2` = Optic disc, confirmed against
the CrossFiT reference loader (`reference/CrossFiT/CrossFiT/dataset.py`), which is DRTiD's own
benchmark authors' code.

**Important scope correction (verified directly against the raw CSVs, not assumed):** every `ID`
value in DRTiD's ground-truth files appears **exactly once** — no `ID` has both an `L` and `R`
row. DRTiD's public metadata exposes a per-**eye** identifier, not a separate patient identifier
linking two eyes back to the same person. The split below (and the bootstrap in Section 17) group
by this `ID` because it's the finest-grained key the released data provides — but that is **not**
the same as a verified patient-wise split. It's possible (not ruled out by anything in the public
CSVs) that both eyes of the same real patient land in different splits. State this explicitly as a
limitation in the paper rather than claiming "patient-wise" without qualification — this is
exactly the kind of unverified claim `judge.md` warns against.

In [ ]:
from sklearn.model_selection import train_test_split

def make_drtid_splits(seed=42, val_fraction=0.2, force=False):
    train_out = f"{SPLITS_DIR}/drtid_train.csv"
    val_out   = f"{SPLITS_DIR}/drtid_val.csv"
    test_out  = f"{SPLITS_DIR}/drtid_test.csv"

    if not force and all(os.path.exists(p) for p in [train_out, val_out, test_out]):
        print("Splits already exist on Drive, skipping regeneration (set force=True to redo).")
        return train_out, val_out, test_out

    images_dir = f"{DRTID_ROOT}/Original Images"
    off_train = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv")
    off_test  = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv")

    overlap = set(off_train["ID"]) & set(off_test["ID"])
    assert not overlap, f"Gate 1 FAILED: patient overlap between official train/test: {overlap}"

    def standardize(df):
        return pd.DataFrame({
            "patient_id": df["ID"],
            "macula_path": df["Macula"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "disc_path": df["Optic disc"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "grade": df["Grade"],
        })

    train_ids, val_ids = train_test_split(off_train["ID"].values, test_size=val_fraction, random_state=seed)
    train_df = standardize(off_train[off_train["ID"].isin(train_ids)])
    val_df   = standardize(off_train[off_train["ID"].isin(val_ids)])
    test_df  = standardize(off_test)

    assert not (set(train_df.patient_id) & set(val_df.patient_id)), "Gate 1 FAILED: train/val overlap"
    assert not (set(val_df.patient_id) & set(test_df.patient_id)), "Gate 1 FAILED: val/test overlap"

    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        missing = [p for col in ("macula_path", "disc_path") for p in df[col] if not os.path.exists(p)]
        assert not missing, f"Gate 1 FAILED: {len(missing)} missing image(s) in {name}, e.g. {missing[:3]}"
        dist = df["grade"].value_counts().sort_index()
        print(f"[{name}] n={len(df)} grade dist: " + ", ".join(f"G{g}={c}" for g, c in dist.items()))
        missing_grades = set(range(5)) - set(dist.index)
        if missing_grades:
            print(f"  WARNING: grades {sorted(missing_grades)} absent from {name}")

    train_df.to_csv(train_out, index=False)
    val_df.to_csv(val_out, index=False)
    test_df.to_csv(test_out, index=False)
    print(f"\nGate 1: PASSED. Wrote splits to {SPLITS_DIR}")
    return train_out, val_out, test_out

DRTID_TRAIN_CSV, DRTID_VAL_CSV, DRTID_TEST_CSV = make_drtid_splits(seed=42)

## 5. Datasets & transforms

Horizontal flip is deliberately **omitted**, not just defaulted off: it risks changing the
clinical meaning of macula/disc laterality (left vs right eye), and the CrossFiT reference
implementation itself has flip code present but commented out -- i.e. DRTiD's own benchmark
authors made the same call.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# DRTiD-specific channel stats (from reference/CrossFiT/CrossFiT/dataset.py -- the CrossFiT
# authors' own computed stats on this exact dataset). Keeps preprocessing aligned with the
# benchmark this project is compared against.
DRTID_MEAN = [0.372487, 0.217266, 0.119367]
DRTID_STD  = [0.281526, 0.179457, 0.109162]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(train: bool, mean, std) -> A.Compose:
    if train:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Rotate(limit=15, p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

train_transform = build_transforms(True, DRTID_MEAN, DRTID_STD)
eval_transform  = build_transforms(False, DRTID_MEAN, DRTID_STD)
aptos_train_transform = build_transforms(True, IMAGENET_MEAN, IMAGENET_STD)
aptos_eval_transform  = build_transforms(False, IMAGENET_MEAN, IMAGENET_STD)

def _load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

class DRTiDDualViewDataset(Dataset):
    def __init__(self, split_csv, transform=None):
        self.df = pd.read_csv(split_csv)
        self.transform = transform if transform is not None else eval_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        macula = self.transform(image=_load_rgb(row["macula_path"]))["image"]
        disc = self.transform(image=_load_rgb(row["disc_path"]))["image"]
        return {"macula": macula, "disc": disc,
                "label": torch.tensor(int(row["grade"]), dtype=torch.long),
                "patient_id": row["patient_id"]}

class APTOSSingleViewDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.transform = transform if transform is not None else aptos_eval_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = _load_rgb("{}/{}.png".format(self.root_dir, row["id_code"]))
        img = self.transform(image=img)["image"]
        return {"image": img, "label": torch.tensor(int(row["diagnosis"]), dtype=torch.long)}

print("Datasets defined.")

## 6. Models

`CORALHead` guarantees monotonic cumulative-probability outputs **by construction**
(ordered-bias parameterization) -- not left to training to discover (judge.md Flag 16 /
technical doc Section 3.1). Teacher and student both expose `forward_single()` (honest
single-view baselines) and `counterfactual_forward()` -- the same-head counterfactual
formulation that is judge.md's most important fix (Flag 1 / Flag 3): macula-only, disc-only
and dual predictions all go through the *same* `main_head`, so their difference cannot be
attributed to head-to-head parameter/calibration discrepancy the way the default
macula_head/disc_head/main_head comparison can.

In [ ]:
import torchvision.models as tv

class CORALHead(nn.Module):
    """Ordinal head with monotone cumulative outputs P(y>k), k=0..K-2.

    Monotonicity is guaranteed by construction: thresholds are base_bias minus a cumulative sum
    of NON-NEGATIVE softplus steps, so they are non-increasing in k no matter what training does.

    REV3 FIX -- threshold initialization (highest-impact change in this revision).
    v2 initialized bias_steps at a constant -3.0, giving softplus(-3)=0.049 per step, i.e. the
    four thresholds all started at P(y>k) between 0.46 and 0.50 -- essentially collapsed on top of
    each other. DRTiD's actual marginals need thresholds spread across logits
    [+0.03, -0.33, -1.55, -3.21], a span of 3.24; v2 started with a span of 0.15, ~22x too
    compressed. Because CORAL's shared-weight design gives each sample only ONE scalar score
    g(z) that gets compared against all four thresholds, near-identical thresholds mean the
    prediction jumps from grade 0 straight to grade 4 with almost nothing in between -- which is
    exactly the pathology measured in the v2 run (Grade1/2/3 sensitivity ~0.00-0.03 across every
    single condition, teacher included).

    Passing init_thresholds=compute_init_thresholds(train_csv) starts the model AT the dataset's
    empirical cumulative distribution, so it only has to learn deviations from the marginal rather
    than first having to discover the marginal itself from a degenerate start. This is standard
    practice for ordinal/threshold models and is not tuning on validation data -- it reads only
    training-split label frequencies.
    """

    def __init__(self, in_dim, num_classes=NUM_CLASSES, init_thresholds=None):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)

        if init_thresholds is None:
            # Fallback: evenly spaced by 0.7 logits (roughly the spacing softplus(0) gives).
            init_thresholds = [-0.7 * i for i in range(self.num_thresholds)]
        t = torch.tensor(list(init_thresholds), dtype=torch.float32)
        if t.numel() != self.num_thresholds:
            raise ValueError(f"init_thresholds must have {self.num_thresholds} entries, got {t.numel()}")
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)          # positive gaps between consecutive thresholds
        inv_softplus_gaps = torch.log(torch.expm1(gaps))  # softplus(x)=gap  =>  x=log(exp(gap)-1)

        self.base_bias = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(inv_softplus_gaps.clone())

    def _ordered_biases(self):
        steps = F.softplus(self.bias_steps)
        cum = torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, dim=0)])
        return self.base_bias - cum

    def forward(self, z):
        g = self.fc(z)
        biases = self._ordered_biases().unsqueeze(0)
        logits = g + biases
        probas = torch.sigmoid(logits)  # P(y>k), guaranteed non-increasing in k
        return logits, probas


class InteractionFusion(nn.Module):
    # judge.md Flag 2 fix, now empirically motivated too: a bare Linear-on-concatenation fusion
    # (the technical doc's original "Concatenation -> FC") has no way to represent multiplicative
    # or differential interaction between the two views -- only a weighted sum. Confirmed on a
    # real DRTiD teacher run: the resulting dual-view head UNDERPERFORMED its own macula-only
    # auxiliary head (QWK_dual=0.558 < QWK_macula=0.572, Gate 2 failure -- see
    # experiment/notebook-result/result-note.md). This adds the interaction terms judge.md
    # explicitly recommended (|z_m - z_d|, z_m * z_d) through a small MLP.
    #
    # Also replaces BatchNorm1d with LayerNorm: BatchNorm statistics computed from a batch of 8
    # (the teacher's batch size) are a known noise source, and a plausible contributor to the
    # volatile epoch-to-epoch QWK swings seen even during the frozen-backbone phase of that same
    # run, when only this fusion block + heads were training. LayerNorm has no batch-size
    # dependence.
    #
    # fusion_type="linear" keeps the original bare-linear+norm behavior available as an ablation
    # (judge.md's own suggested ablation set: linear / MLP / product-difference features) rather
    # than removing it outright.
    def __init__(self, feat_dim, fusion_type="interaction_mlp", hidden_dim=None):
        super().__init__()
        if fusion_type not in ("linear", "interaction_mlp"):
            raise ValueError(f"unknown fusion_type: {fusion_type}")
        self.fusion_type = fusion_type
        hidden_dim = hidden_dim if hidden_dim is not None else feat_dim

        # REV3 FIX (TorchScript). v2 created self.norm ONLY when fusion_type=="linear" and the
        # MLP submodules only when fusion_type=="interaction_mlp", but forward() references both
        # branches. torch.jit.script statically analyses EVERY branch regardless of which one can
        # actually execute, so scripting the deployed (interaction_mlp) model failed with
        # "Module 'InteractionFusion' has no attribute 'norm'" -- which is precisely the error the
        # v2 run hit at Gate 5, forcing a state_dict fallback instead of a real TorchScript
        # deployment artifact. Defining all submodules unconditionally costs a few unused
        # parameters and makes the module scriptable.
        self.norm = nn.LayerNorm(feat_dim * 2)
        self.norm_in = nn.LayerNorm(feat_dim * 4)
        self.proj = nn.Linear(feat_dim * 4, hidden_dim)
        self.act = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden_dim)

        self.out_dim = feat_dim * 2 if fusion_type == "linear" else hidden_dim

    def forward(self, z_m, z_d):
        if self.fusion_type == "linear":
            return self.norm(torch.cat([z_m, z_d], dim=1))
        diff = torch.abs(z_m - z_d)
        prod = z_m * z_d
        combined = self.norm_in(torch.cat([z_m, z_d, diff, prod], dim=1))
        return self.norm_out(self.act(self.proj(combined)))


class DualViewResNetTeacher(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=2048, fusion_type="interaction_mlp",
                 init_thresholds=None):
        super().__init__()
        backbone = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.fusion = InteractionFusion(feat_dim, fusion_type=fusion_type)
        self.main_head = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(feat_dim, num_classes, init_thresholds)
        self.disc_head = CORALHead(feat_dim, num_classes, init_thresholds)

    def forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        z_fused = self.fusion(z_m, z_d)
        logit_dual, p_dual = self.main_head(z_fused)
        logit_m, p_m = self.macula_head(z_m)
        logit_d, p_d = self.disc_head(z_d)
        return {"p_dual": p_dual, "logit_dual": logit_dual, "p_macula": p_m, "logit_macula": logit_m,
                "p_disc": p_d, "logit_disc": logit_d, "z_fused": z_fused}

    def forward_single(self, x, which="macula"):
        z = self.backbone(x)
        head = self.macula_head if which == "macula" else self.disc_head
        logit, p = head(z)
        return {"logit": logit, "p": p}

    def counterfactual_forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        zero = torch.zeros_like(z_m)
        _, p_dual = self.main_head(self.fusion(z_m, z_d))
        _, p_m_only = self.main_head(self.fusion(z_m, zero))
        _, p_d_only = self.main_head(self.fusion(zero, z_d))
        return {"p_dual": p_dual, "p_macula_cf": p_m_only, "p_disc_cf": p_d_only}


class DepthwiseSeparableBlock(nn.Module):
    # Each sub-layer is its own named module instance -- required for
    # torch.ao.quantization.fuse_modules (fuses by module identity).
    # ReLU (not ReLU6): eager-mode fuse_modules has no fuser for Conv-BN-ReLU6 (judge.md Code
    # issue 4) -- reproduced and confirmed locally, fixed by using plain ReLU.
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.act1 = nn.ReLU(inplace=True)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.act2 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act1(self.bn1(self.dw(x)))
        x = self.act2(self.bn2(self.pw(x)))
        return x

    def fuse(self):
        torch.ao.quantization.fuse_modules(self, [["dw", "bn1", "act1"], ["pw", "bn2", "act2"]], inplace=True)


class LightweightBackbone(nn.Module):
    """Depthwise-separable student backbone.

    REV3 FIX -- capacity. v2's channel plan (16,24,24,40,40,56) produced a conv stack with only
    ~8,176 parameters; the InteractionFusion MLP was 75% of the entire 34K-parameter student. The
    project's own technical documentation (Appendix C item 6) targets ~0.3-0.4M parameters and
    flags under-capacity as a risk to watch. An 8K-parameter feature extractor for 224x224 fundus
    images is far below that, and it shows in the v2 results: dual_no_distill (0.447),
    dual_logitkd (0.415) and dual_csd (0.406) all landed within ~0.04 QWK of each other with
    overlapping seed spreads -- what a shared capacity ceiling looks like. If the student cannot
    represent the task any better regardless of supervision, no distillation signal can show a
    benefit, and RQ1 becomes untestable rather than answered.

    The default STUDENT_CHANNELS plan brings the whole student to ~330K parameters (still ~170x
    smaller than the 57.1M-parameter ResNet-50 teacher, so the efficiency claim stays strong)
    while giving distillation real headroom to act on.
    """

    def __init__(self, channels=None):
        super().__init__()
        channels = tuple(channels if channels is not None else STUDENT_CHANNELS)
        self.stem_conv = nn.Conv2d(3, channels[0], 3, stride=2, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(channels[0])
        self.stem_act = nn.ReLU(inplace=True)
        # Alternating stride-2 / stride-1 so spatial size shrinks 224 -> 7 across the stack.
        strides = [2 if i % 2 == 0 else 1 for i in range(len(channels) - 1)]
        self.blocks = nn.ModuleList([
            DepthwiseSeparableBlock(channels[i], channels[i + 1], stride=strides[i])
            for i in range(len(channels) - 1)
        ])
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.out_dim = channels[-1]

    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for block in self.blocks:
            x = block(x)
        return self.gap(x).flatten(1)

    def fuse_model(self):
        torch.ao.quantization.fuse_modules(self, [["stem_conv", "stem_bn", "stem_act"]], inplace=True)
        for block in self.blocks:
            block.fuse()


class DualViewLightStudent(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, backbone=None, fusion_type="interaction_mlp",
                 init_thresholds=None):
        super().__init__()
        self.backbone = backbone if backbone is not None else LightweightBackbone()
        feat_dim = self.backbone.out_dim
        self.fusion = InteractionFusion(feat_dim, fusion_type=fusion_type)
        self.main_head = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(feat_dim, num_classes, init_thresholds)
        self.disc_head = CORALHead(feat_dim, num_classes, init_thresholds)

    def forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        z_fused = self.fusion(z_m, z_d)
        logit_dual, p_dual = self.main_head(z_fused)
        logit_m, p_m = self.macula_head(z_m)
        logit_d, p_d = self.disc_head(z_d)
        return {"p_dual": p_dual, "logit_dual": logit_dual, "p_macula": p_m, "logit_macula": logit_m,
                "p_disc": p_d, "logit_disc": logit_d, "z_fused": z_fused}

    def forward_single(self, x, which="macula"):
        z = self.backbone(x)
        head = self.macula_head if which == "macula" else self.disc_head
        logit, p = head(z)
        return {"logit": logit, "p": p}

    def counterfactual_forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        zero = torch.zeros_like(z_m)
        _, p_dual = self.main_head(self.fusion(z_m, z_d))
        _, p_m_only = self.main_head(self.fusion(z_m, zero))
        _, p_d_only = self.main_head(self.fusion(zero, z_d))
        return {"p_dual": p_dual, "p_macula_cf": p_m_only, "p_disc_cf": p_d_only}

    def fuse_model(self):
        if hasattr(self.backbone, "fuse_model"):
            self.backbone.fuse_model()

print("Models defined.")

## 7. Losses

`csd_loss` supports 3 variants (`smoothl1` default, `direction_magnitude`, `kl_softmax`
ablation-only). `csd_loss_no_aux_gradient` detaches student aux outputs so CSD cannot be
minimized by drifting the auxiliary heads instead of improving the dual head (judge.md Flag 3).
`get_student_output` is the single place `view_mode` selects the forward path, for both
training and evaluation, so single-view baselines can never silently train the dual head
(technical doc Critical Issue 1).

In [ ]:
def coral_loss(logits, labels, num_thresholds=NUM_THRESHOLDS, pos_weight=None):
    device = logits.device
    levels = torch.arange(num_thresholds, device=device).unsqueeze(0)
    y_k = (labels.unsqueeze(1) > levels).float()
    return F.binary_cross_entropy_with_logits(logits, y_k, pos_weight=pos_weight)

def aux_loss(student_out, labels, num_thresholds=NUM_THRESHOLDS, pos_weight=None):
    l_m = coral_loss(student_out["logit_macula"], labels, num_thresholds, pos_weight=pos_weight)
    l_d = coral_loss(student_out["logit_disc"], labels, num_thresholds, pos_weight=pos_weight)
    return l_m + l_d

def logit_kd_loss(logit_dual_teacher, logit_dual_student, tau=2.0):
    p_t = torch.sigmoid(logit_dual_teacher.detach() / tau)
    p_s = torch.sigmoid(logit_dual_student / tau)
    return F.binary_cross_entropy(p_s, p_t)

def _compute_delta(p_dual, p_macula, p_disc):
    return p_dual - (p_macula + p_disc) / 2

def csd_loss(p_dual_t, p_macula_t, p_disc_t, p_dual_s, p_macula_s, p_disc_s,
             variant="smoothl1_norm", tau_csd=0.5, huber_beta=1.0, eps=1e-6):
    """Complementarity-Shift Distillation loss.

    REV3 FIX -- loss scale (the mechanistic cause of v2's null result).
    Delta lives in cumulative-probability space, so its per-dimension entries are small: the v2
    run logged L_CSD ~= 0.014 against L_task ~= 0.82 and lambda_aux*L_aux ~= 0.86. At the
    grid-selected beta=0.7 that is under 0.5% of total loss, i.e. CSD was contributing
    essentially no gradient and RQ1 was really testing "does adding a near-zero term change
    anything" (it does not). Worse, plain SmoothL1 with the default huber_beta=1.0 keeps small
    residuals in its QUADRATIC regime, where d/dx (0.5x^2) = x -- so a residual of 0.05 yields a
    gradient of 0.05, shrinking the signal a second time. judge.md Flag 6 predicted exactly this
    ("kontribusi gradient CSD mungkin hampir nol") before any training was run.

    'smoothl1_norm' (new default) divides both shift vectors by the teacher shift's own mean
    magnitude (detached, per batch) before the Huber loss. This makes the loss scale-free and
    O(1), so beta now controls a meaningful contribution instead of being swamped, WITHOUT
    discarding magnitude information the way softmax normalization does. Because the divisor is
    detached and identical for both sides, it rescales the objective without changing which
    student shift pattern is optimal.

    Variants
      'smoothl1_norm'                 -- DEFAULT. Scale-normalized signed Huber on Delta.
      'smoothl1'                      -- v2 behavior, kept for a controlled before/after ablation.
      'magnitude_weighted_direction'  -- judge.md Flag 5: cosine-direction term weighted by the
                                         teacher's shift magnitude, so samples whose teacher shift
                                         is near zero (where direction is meaningless noise) do not
                                         dominate; plus a normalized magnitude term.
      'kl_softmax'                    -- v1 formulation, ABLATION ONLY. The v2 run logged
                                         L_CSD == 0.0000 for all 16 epochs, empirically confirming
                                         that softmax-normalizing Delta destroys the magnitude
                                         information CSD is supposed to transfer.
    """
    delta_t = _compute_delta(p_dual_t.detach(), p_macula_t.detach(), p_disc_t.detach())
    delta_s = _compute_delta(p_dual_s, p_macula_s, p_disc_s)

    if variant == "smoothl1_norm":
        scale = delta_t.abs().mean().detach().clamp_min(1e-3)
        return F.smooth_l1_loss(delta_s / scale, delta_t / scale, beta=huber_beta)
    elif variant == "smoothl1":
        return F.smooth_l1_loss(delta_s, delta_t, beta=huber_beta)
    elif variant == "magnitude_weighted_direction":
        mag = delta_t.norm(dim=1)                                   # [B]
        w = (mag / mag.median().clamp_min(eps)).clamp(max=1.0)      # down-weight near-zero shifts
        cos_sim = F.cosine_similarity(delta_s, delta_t, dim=1, eps=eps)
        l_dir = ((1.0 - cos_sim) * w).sum() / w.sum().clamp_min(eps)
        scale = delta_t.abs().mean().detach().clamp_min(1e-3)
        l_mag = F.smooth_l1_loss(delta_s / scale, delta_t / scale, beta=huber_beta)
        return 0.5 * l_dir + 0.5 * l_mag
    elif variant == "direction_magnitude":
        cos_sim = F.cosine_similarity(delta_s, delta_t, dim=1, eps=eps)
        l_dir = (1 - cos_sim).mean()
        l_mag = F.smooth_l1_loss(delta_s, delta_t, beta=huber_beta)
        return 0.5 * l_dir + 0.5 * l_mag
    elif variant == "kl_softmax":
        log_q = F.log_softmax(delta_s / tau_csd, dim=1)
        p_target = F.softmax(delta_t / tau_csd, dim=1)
        return F.kl_div(log_q, p_target, reduction="batchmean")
    raise ValueError(f"unknown csd_variant: {variant}")


def feature_kd_loss(z_fused_t, z_fused_s, projector):
    """judge.md Section G item 3 -- feature-KD control baseline.

    Projects the teacher's fused representation into the student's fused space and matches them
    with MSE. This is the key control for CSD's central claim: if CSD only matches generic
    feature-level KD, then "distilling the decision-shift pattern" is not doing anything special
    beyond ordinary representation transfer. If CSD beats it, the claim that DECISION-level shift
    knowledge is more transferable than feature-level similarity gains real evidence.
    """
    target = projector(z_fused_t.detach())
    return F.mse_loss(z_fused_s, target)

def get_student_output(student, macula, disc, view_mode):
    if view_mode == "dual":
        return student(macula, disc)
    elif view_mode == "macula_only":
        return student.forward_single(macula, which="macula")
    elif view_mode == "disc_only":
        return student.forward_single(disc, which="disc")
    raise ValueError(f"unknown view_mode: {view_mode}")

def ordinal_violation_rate(p):
    diffs = p[:, 1:] - p[:, :-1]
    return (diffs > 0).float().mean().item()

def combined_student_loss(teacher_out, student_out, labels, view_mode, alpha=0.0, beta=0.0,
                           lambda_aux=0.5, tau_kd=2.0, csd_variant="smoothl1_norm", tau_csd=0.5,
                           pos_weight=None, use_counterfactual_csd=False,
                           teacher_cf_out=None, student_cf_out=None,
                           gamma_feat=0.0, feat_projector=None, huber_beta=1.0):
    """Returns (total_loss, scalar_log_dict, component_tensor_dict).

    REV3 CHANGE: also returns the individual WEIGHTED loss tensors so the training loop can
    measure each component's gradient norm (judge.md Flag 6). Reporting only loss VALUES is not
    enough to establish that a term influences learning -- two terms with similar magnitudes can
    have very different gradient norms. v2 reported values only, which is why the CSD-contributes-
    nothing problem was invisible until the logs were re-read by hand after the whole run.
    """
    task_logit = student_out["logit_dual"] if view_mode == "dual" else student_out["logit"]
    l_task = coral_loss(task_logit, labels, pos_weight=pos_weight)
    total = l_task
    log = {"L_task": l_task.item()}
    components = {"task": l_task}

    if view_mode == "dual":
        l_aux = aux_loss(student_out, labels, pos_weight=pos_weight)
        total = total + lambda_aux * l_aux
        log["L_aux"] = l_aux.item()
        components["aux"] = lambda_aux * l_aux

        if alpha > 0:
            l_kd = logit_kd_loss(teacher_out["logit_dual"], student_out["logit_dual"], tau_kd)
            total = total + alpha * l_kd
            log["L_logit_KD"] = l_kd.item()
            components["logit_kd"] = alpha * l_kd

        if beta > 0:
            if use_counterfactual_csd:
                l_csd = csd_loss(teacher_cf_out["p_dual"], teacher_cf_out["p_macula_cf"], teacher_cf_out["p_disc_cf"],
                                  student_cf_out["p_dual"], student_cf_out["p_macula_cf"], student_cf_out["p_disc_cf"],
                                  variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta)
            else:
                l_csd = csd_loss(teacher_out["p_dual"], teacher_out["p_macula"], teacher_out["p_disc"],
                                  student_out["p_dual"], student_out["p_macula"], student_out["p_disc"],
                                  variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta)
            total = total + beta * l_csd
            log["L_CSD"] = l_csd.item()
            components["csd"] = beta * l_csd

        if gamma_feat > 0 and feat_projector is not None:
            l_feat = feature_kd_loss(teacher_out["z_fused"], student_out["z_fused"], feat_projector)
            total = total + gamma_feat * l_feat
            log["L_feat_KD"] = l_feat.item()
            components["feat_kd"] = gamma_feat * l_feat

    log["L_total"] = total.item()
    return total, log, components


def component_grad_norms(components, params):
    """L2 norm of each WEIGHTED loss component's gradient w.r.t. `params` (judge.md Flag 6).

    This is what actually settles "is CSD influencing training?" -- a term can have a tiny loss
    value but a meaningful gradient, or vice versa. Called on one batch per epoch only, since
    each component needs its own autograd.grad pass and that is not free.
    """
    out = {}
    params = [p for p in params if p.requires_grad]
    for name, loss_tensor in components.items():
        if loss_tensor is None or not loss_tensor.requires_grad:
            continue
        grads = torch.autograd.grad(loss_tensor, params, retain_graph=True, allow_unused=True)
        sq = 0.0
        for g in grads:
            if g is not None:
                sq += float(g.pow(2).sum().item())
        out[f"gnorm_{name}"] = sq ** 0.5
    # Ratio of CSD's gradient to the task gradient -- the single number that says whether CSD is
    # a real training signal or numerical decoration.
    if "gnorm_csd" in out and out.get("gnorm_task", 0.0) > 0:
        out["gnorm_ratio_csd_over_task"] = out["gnorm_csd"] / out["gnorm_task"]
    return out

print("Losses defined.")

## 8. Smoke test — must pass before any real training

In [ ]:
INIT_THRESHOLDS = compute_init_thresholds(DRTID_TRAIN_CSV)
print("CORAL init thresholds (logit of empirical P(y>k)):",
      [round(t, 4) for t in INIT_THRESHOLDS])
print("  -> implied initial P(y>k):",
      [round(float(torch.sigmoid(torch.tensor(t))), 4) for t in INIT_THRESHOLDS])
POS_WEIGHT = compute_pos_weights(DRTID_TRAIN_CSV)
print(f"pos_weight (mode={POS_WEIGHT_MODE}):", [round(float(w), 3) for w in POS_WEIGHT])


def smoke_test():
    ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, transform=eval_transform)
    loader = DataLoader(ds, batch_size=8, shuffle=True)
    batch = next(iter(loader))
    macula, disc, y = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE), batch["label"].to(DEVICE)

    teacher = DualViewResNetTeacher(init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student = DualViewLightStudent(init_thresholds=INIT_THRESHOLDS).to(DEVICE)

    n_student = sum(p.numel() for p in student.parameters())
    n_backbone = sum(p.numel() for p in student.backbone.parameters())
    n_teacher = sum(p.numel() for p in teacher.parameters())
    print(f"Student params: {n_student:,} (backbone {n_backbone:,}) | Teacher params: {n_teacher:,} "
          f"| compression {n_teacher / n_student:.0f}x")
    assert n_student > 150_000, (
        f"REV3 capacity check FAILED: student has only {n_student:,} params -- the v2 run's 34K "
        f"student was capacity-capped and made RQ1 untestable. Check STUDENT_CHANNELS."
    )

    teacher_out = teacher(macula, disc)
    ovr = ordinal_violation_rate(teacher_out["p_dual"])
    assert ovr == 0.0, f"CORAL monotonicity FAILED, OVR={ovr}"
    print(f"Teacher forward OK, OVR={ovr}")

    # REV3 check: thresholds must actually be SPREAD, not collapsed. The v2 initialization put all
    # four within 0.15 logits of each other, which forced predictions to jump grade 0 -> grade 4.
    biases = teacher.main_head._ordered_biases().detach().cpu()
    spread = float(biases[0] - biases[-1])
    print(f"CORAL threshold spread (logits): {spread:.3f}  thresholds={[round(float(b),3) for b in biases]}")
    assert spread > 1.5, (
        f"REV3 threshold-init check FAILED: spread {spread:.3f} is too small; predictions will "
        f"collapse onto the extreme grades as they did in v2."
    )

    _ = teacher.forward_single(macula, which="macula")
    cf_out = teacher.counterfactual_forward(macula, disc)
    print("Teacher forward_single / counterfactual_forward OK")

    for view_mode in ["dual", "macula_only", "disc_only"]:
        student_out = get_student_output(student, macula, disc, view_mode)
        loss, log, comps = combined_student_loss(teacher_out, student_out, y, view_mode,
                                                  alpha=0.5, beta=1.0, pos_weight=POS_WEIGHT.to(DEVICE))
        loss.backward()
        student.zero_grad()
        print(f"[{view_mode}] OK -- loss={loss.item():.4f}")

    student_cf_out = student.counterfactual_forward(macula, disc)
    student_out_dual = student(macula, disc)
    loss_cf, log_cf, _ = combined_student_loss(teacher_out, student_out_dual, y, "dual", alpha=0.5, beta=1.0,
                                                use_counterfactual_csd=True, teacher_cf_out=cf_out,
                                                student_cf_out=student_cf_out, pos_weight=POS_WEIGHT.to(DEVICE))
    loss_cf.backward()
    student.zero_grad()
    print(f"[dual, counterfactual CSD] OK -- loss={loss_cf.item():.4f}")

    # REV3 check: the CSD term must produce a NON-NEGLIGIBLE gradient relative to the task loss.
    # This is the exact failure mode that silently invalidated v2's RQ1 test (judge.md Flag 6).
    student_out_dual = student(macula, disc)
    _, log_g, comps_g = combined_student_loss(teacher_out, student_out_dual, y, "dual",
                                               alpha=0.5, beta=1.0, pos_weight=POS_WEIGHT.to(DEVICE))
    fusion_params = list(student.fusion.parameters()) + list(student.main_head.parameters())
    gnorms = component_grad_norms(comps_g, fusion_params)
    student.zero_grad()
    print("Component gradient norms (fusion+main_head):",
          {k: round(v, 5) for k, v in gnorms.items()})
    ratio = gnorms.get("gnorm_ratio_csd_over_task", 0.0)
    assert ratio > 0.01, (
        f"REV3 CSD-scale check FAILED: CSD/task gradient ratio is {ratio:.5f}. In the v2 run CSD "
        f"contributed <0.5% of the loss and its gradient was negligible, so RQ1 was never really "
        f"tested. Increase beta or check the csd_variant normalization."
    )
    print(f"CSD gradient is a real training signal at beta=1.0 (CSD/task grad ratio = {ratio:.4f}) -- OK")
    print("  NOTE: this probe uses beta=1.0 to make the signal obvious. The GRID in Section 14 "
          "searches beta in [0.1, 0.5] because a ratio far above ~1 means CSD would start "
          "overwhelming the task loss -- the opposite of v2's failure, and just as damaging.")

    student.eval()  # Conv-BN fusion requires eval mode (torch's own fuse_conv_bn_eval assertion)
    student.fuse_model()  # confirms PTQ fusion works before Day 8, not after
    print("fuse_model() OK")

    print("\nSMOKE TEST PASSED.")

smoke_test()

## 9. Evaluation helpers

`measure_cpu_latency` always runs against a CPU-copied model (judge.md Code issue 1: the
original design measured GPU latency but labeled the column "CPU_Latency" -- fixed here by
construction, there is no code path that can mislabel GPU timing as CPU timing).

In [ ]:
import time, copy
from sklearn.metrics import cohen_kappa_score, f1_score, recall_score

@torch.no_grad()
def quick_val_qwk(model, loader, device, view_mode="dual"):
    # Fast QWK-only check used INSIDE training loops for best-checkpoint selection.
    model.eval()
    preds, targets = [], []
    for batch in loader:
        macula, disc, y = batch["macula"].to(device), batch["disc"].to(device), batch["label"]
        out = get_student_output(model, macula, disc, view_mode) if hasattr(model, "forward_single") else model(macula, disc)
        p_key = "p_dual" if view_mode == "dual" else "p"
        grade_pred = (out[p_key] > 0.5).sum(dim=1)
        preds.extend(grade_pred.cpu().tolist())
        targets.extend(y.tolist())
    return cohen_kappa_score(targets, preds, weights="quadratic")

@torch.no_grad()
def get_predictions(model, loader, device, view_mode="dual"):
    model.eval()
    preds, targets, all_p = [], [], []
    for batch in loader:
        macula, disc, y = batch["macula"].to(device), batch["disc"].to(device), batch["label"]
        out = get_student_output(model, macula, disc, view_mode)
        p_key = "p_dual" if view_mode == "dual" else "p"
        grade_pred = (out[p_key] > 0.5).sum(dim=1)
        preds.extend(grade_pred.cpu().tolist())
        targets.extend(y.tolist())
        all_p.append(out[p_key].cpu())
    return np.array(targets), np.array(preds), torch.cat(all_p, dim=0)

def compute_metrics(y_true, y_pred, p_cumulative=None):
    metrics = {
        "QWK": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "MAE": float(np.mean(np.abs(y_true - y_pred))),
        "SevereErrorRate": float(np.mean(np.abs(y_true - y_pred) >= 2)),
        "MacroF1": f1_score(y_true, y_pred, average="macro"),
    }
    sens = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3, 4])
    for g in range(5):
        metrics[f"Sensitivity_Grade{g}"] = sens[g]
    if p_cumulative is not None:
        metrics["OrdinalViolationRate"] = ordinal_violation_rate(p_cumulative)
    return metrics

@torch.no_grad()
def compute_dual_view_gain(model, loader, device):
    # INTERNAL gain: dual head vs this same model's own auxiliary heads (judge.md Flag 8 --
    # explicitly labeled internal, not conflated with an external independently-trained gain).
    y_true, pred_dual, p_dual = get_predictions(model, loader, device, "dual")
    _, pred_macula, _ = get_predictions(model, loader, device, "macula_only")
    _, pred_disc, _ = get_predictions(model, loader, device, "disc_only")
    qwk_dual = cohen_kappa_score(y_true, pred_dual, weights="quadratic")
    qwk_macula = cohen_kappa_score(y_true, pred_macula, weights="quadratic")
    qwk_disc = cohen_kappa_score(y_true, pred_disc, weights="quadratic")
    return {"QWK_dual": qwk_dual, "QWK_macula": qwk_macula, "QWK_disc": qwk_disc,
            "DualViewGain_G_internal": qwk_dual - max(qwk_macula, qwk_disc)}


def _sample_ordinal_nll(p_cum, y, num_thresholds=NUM_THRESHOLDS):
    """Per-sample ordinal negative log-likelihood from cumulative probabilities."""
    levels = torch.arange(num_thresholds, device=p_cum.device).unsqueeze(0)
    y_k = (y.unsqueeze(1) > levels).float()
    p = p_cum.clamp(1e-6, 1 - 1e-6)
    return -(y_k * torch.log(p) + (1 - y_k) * torch.log(1 - p)).sum(dim=1)


@torch.no_grad()
def compute_shift_fidelity(teacher, student, loader, device):
    """judge.md Flag 10 -- direct evidence about whether CSD actually transfers the shift.

    QWK alone cannot establish that the complementarity-shift PATTERN was transferred: a student
    could improve QWK for unrelated reasons, or match the teacher's shift while gaining nothing.
    v2 reported no such metric, so "did CSD do what it claims?" was unanswerable independently of
    "did QWK go up?". These three do answer it:

      ShiftMAE   -- mean L1 distance between student and teacher shift vectors (lower = closer).
      CosAgree   -- mean cosine similarity of the shift DIRECTIONS (higher = same pattern).
      BenefitCorr-- Pearson correlation between the teacher's and student's per-sample
                    "fusion benefit" B_i = NLL(p_agg) - NLL(p_dual). If CSD genuinely transfers
                    complementarity, the student should benefit from dual-view on roughly the
                    SAME samples the teacher does, which is a much stronger claim than a QWK delta.
    """
    teacher.eval(); student.eval()
    shift_mae, cos_agree, ben_t, ben_s = [], [], [], []
    for batch in loader:
        macula, disc = batch["macula"].to(device), batch["disc"].to(device)
        y = batch["label"].to(device)
        t_out, s_out = teacher(macula, disc), student(macula, disc)

        dt = _compute_delta(t_out["p_dual"], t_out["p_macula"], t_out["p_disc"])
        ds = _compute_delta(s_out["p_dual"], s_out["p_macula"], s_out["p_disc"])
        shift_mae.append((ds - dt).abs().sum(dim=1).cpu())
        cos_agree.append(F.cosine_similarity(ds, dt, dim=1, eps=1e-6).cpu())

        for out, sink in ((t_out, ben_t), (s_out, ben_s)):
            p_agg = (out["p_macula"] + out["p_disc"]) / 2
            sink.append((_sample_ordinal_nll(p_agg, y) - _sample_ordinal_nll(out["p_dual"], y)).cpu())

    shift_mae = torch.cat(shift_mae); cos_agree = torch.cat(cos_agree)
    bt, bs = torch.cat(ben_t), torch.cat(ben_s)
    if bt.std() > 1e-8 and bs.std() > 1e-8:
        benefit_corr = float(((bt - bt.mean()) * (bs - bs.mean())).mean() / (bt.std() * bs.std()))
    else:
        benefit_corr = float("nan")
    return {"ShiftMAE": float(shift_mae.mean()), "CosAgree": float(cos_agree.mean()),
            "BenefitCorr": benefit_corr}


def compute_external_gain(qwk_dual, qwk_indep_macula, qwk_indep_disc):
    """judge.md Flag 8 -- EXTERNAL dual-view gain.

    Internal gain compares a dual model's fusion head against its OWN auxiliary heads, which share
    a backbone trained jointly with the fusion task -- so it is not a comparison against a model
    that only ever saw one view. External gain compares against the independently trained
    macula_only / disc_only students. The two answer different questions and judge.md is explicit
    that they must not be conflated; v2 reported only the internal one.
    """
    return qwk_dual - max(qwk_indep_macula, qwk_indep_disc)


def compute_calibration(p_cum, y_true, n_bins=10):
    """judge.md Flag 4 -- is the weighted-BCE output still interpretable as a probability?

    CSD is framed as distilling a shift in model CONFIDENCE, so how well-calibrated those
    cumulative outputs are is directly load-bearing for the claim. Pools all K-1 thresholds and
    reports Brier score plus expected calibration error. v2 computed neither.
    """
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(y_true).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    target = (y.unsqueeze(1) > levels).float()
    p_flat, t_flat = p.flatten(), target.flatten()

    brier = float(((p_flat - t_flat) ** 2).mean())
    ece, n = 0.0, p_flat.numel()
    edges = torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (p_flat > lo) & (p_flat <= hi) if i > 0 else (p_flat >= lo) & (p_flat <= hi)
        if mask.sum() == 0:
            continue
        ece += float(mask.sum()) / n * abs(float(p_flat[mask].mean()) - float(t_flat[mask].mean()))
    return {"Brier": brier, "ECE": ece}


def model_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

def param_count(model):
    return sum(p.numel() for p in model.parameters())

def measure_cpu_latency(model_gpu, sample_macula, sample_disc, view_mode="dual", n_runs=50, warmup=10, n_threads=1):
    torch.set_num_threads(n_threads)
    model_cpu = copy.deepcopy(model_gpu).to("cpu").eval()
    m_cpu, d_cpu = sample_macula[:1].cpu(), sample_disc[:1].cpu()
    if view_mode == "dual":
        forward_fn = lambda: model_cpu(m_cpu, d_cpu)
    else:
        which = "macula" if "macula" in view_mode else "disc"
        img = m_cpu if which == "macula" else d_cpu
        forward_fn = lambda: model_cpu.forward_single(img, which=which)
    with torch.no_grad():
        for _ in range(warmup):
            forward_fn()
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            forward_fn()
            times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    return {"CPU_Latency_median_ms": float(np.median(times)), "CPU_Latency_p95_ms": float(np.percentile(times, 95))}

print("Evaluation helpers defined.")

## 10. Day 2 — Pretrain APTOS backbones

Two separate runs producing two architecturally different checkpoints (ResNet-50 for the
teacher, lightweight for the student) -- `build_backbone()` is the one place deciding which
architecture a config produces, so a lightweight checkpoint can never end up loaded into the
teacher or vice versa (technical doc Critical Issue 10).

In [ ]:
def build_backbone(backbone_type):
    if backbone_type == "resnet50":
        m = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
        return m, 2048
    elif backbone_type == "lightweight":
        m = LightweightBackbone()
        return m, m.out_dim
    raise ValueError(backbone_type)

def pretrain_backbone(backbone_type, epochs, lr, batch_size, seed=42, force=False):
    out_ckpt = f"{CKPT_DIR}/pretrained_backbones/aptos_{backbone_type}_backbone.pt"
    backbone, feat_dim = build_backbone(backbone_type)
    if not force and checkpoint_is_compatible(out_ckpt, backbone, unwrap_key=None):
        print(f"{out_ckpt} already exists and matches this architecture, skipping (set force=True to redo).")
        return out_ckpt

    set_seed(seed)
    pos_weight = compute_pos_weights(f"{APTOS_ROOT}/train_1.csv", grade_col="diagnosis").to(DEVICE)

    train_ds = APTOSSingleViewDataset(f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/train_images/train_images", aptos_train_transform)
    val_ds = APTOSSingleViewDataset(f"{APTOS_ROOT}/valid.csv", f"{APTOS_ROOT}/val_images/val_images", aptos_eval_transform)
    g = make_generator(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    head = CORALHead(feat_dim, NUM_CLASSES)
    backbone, head = backbone.to(DEVICE), head.to(DEVICE)
    opt = torch.optim.Adam(list(backbone.parameters()) + list(head.parameters()), lr=lr)

    best_qwk, history = -1.0, []
    for epoch in range(epochs):
        backbone.train(); head.train()
        for batch in tqdm(train_loader, desc=f"[pretrain-{backbone_type}] epoch {epoch}"):
            img, y = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
            logit, _ = head(backbone(img))
            loss = coral_loss(logit, y, pos_weight=pos_weight)
            opt.zero_grad(); loss.backward(); opt.step()

        backbone.eval(); head.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                img, y = batch["image"].to(DEVICE), batch["label"]
                _, p = head(backbone(img))
                preds.extend((p > 0.5).sum(dim=1).cpu().tolist())
                targets.extend(y.tolist())
        val_qwk = cohen_kappa_score(targets, preds, weights="quadratic")
        history.append({"epoch": epoch, "val_qwk": val_qwk})
        print(f"epoch {epoch}: val_QWK={val_qwk:.4f}")
        if val_qwk > best_qwk:
            best_qwk = val_qwk
            robust_torch_save(backbone.state_dict(), out_ckpt)

    pd.DataFrame(history).to_csv(f"{LOGS_DIR}/pretrain_{backbone_type}_history.csv", index=False)
    print(f"Pretraining {backbone_type} done. Best val QWK = {best_qwk:.4f} -> {out_ckpt}")
    assert best_qwk > 0.0, f"Gate check: pretrain {backbone_type} val QWK <= 0, worse than majority baseline -- do not proceed."
    return out_ckpt

from tqdm import tqdm

RESNET50_BACKBONE_CKPT = pretrain_backbone("resnet50", epochs=20, lr=1e-4, batch_size=32, seed=42)
LIGHTWEIGHT_BACKBONE_CKPT = pretrain_backbone("lightweight", epochs=30, lr=1e-3, batch_size=32, seed=42)

## 11. Day 3–4 — Teacher training (Gate 2)

Two-stage: freeze backbone + train heads, then unfreeze + fine-tune everything. Checkpointed
on **best val QWK**, not the final epoch (technical doc Critical Issue 9).

In [ ]:
def train_teacher(freeze_epochs=5, finetune_epochs=15, patience=5, lambda_aux=0.3, seed=42,
                   batch_size=16, freeze_lr=3e-4, force=False):
    # lambda_aux lowered 0.5 -> 0.3 and freeze_lr lowered 1e-3 -> 3e-4 after real runs showed the
    # dual head landing right on the edge of beating its own auxiliary heads (Gate 2 passed once
    # at +0.0295 gain, failed twice at ~-0.014 gain across repeated attempts with the
    # InteractionFusion fix already applied -- see experiment/documentation.md Section 0.2).
    # A high lambda_aux gives the single-view aux losses a strong pull on the SHARED backbone
    # (their gradients flow through it during finetune), competing with what's optimal for the
    # fusion task specifically. A high freeze-phase LR was also a plausible contributor: the very
    # first run's freeze-phase val QWK declined for 3 straight epochs right after peaking --
    # consistent with the head-only parameter set overshooting on only 800 training images.
    out_ckpt = f"{CKPT_DIR}/teacher/teacher_final.pt"
    model = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out_ckpt, model, unwrap_key="model_state"):
        print(f"{out_ckpt} already exists and matches this architecture, skipping (set force=True to redo).")
        return out_ckpt
    model = model.to(DEVICE)

    set_seed(seed)
    pos_weight = compute_pos_weights(DRTID_TRAIN_CSV).to(DEVICE)
    train_ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, train_transform)
    val_ds = DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform)
    g = make_generator(seed)
    # batch_size=16 (not 8): a batch of 8 was a plausible contributor to the volatile val-QWK
    # swings seen on the first real run (see experiment/notebook-result/result-note.md) -- larger
    # batches give less noisy gradient estimates on this small (800-image) training set. This is
    # secondary to the InteractionFusion fix (Section 6), which removes fusion's own batch-size
    # dependence entirely by using LayerNorm instead of BatchNorm1d, but there's no reason not to
    # also reduce noise here given Colab Pro GPUs have headroom for it.
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model.backbone.load_state_dict(robust_torch_load(RESNET50_BACKBONE_CKPT, map_location=DEVICE))
    print(f"Teacher backbone loaded from {RESNET50_BACKBONE_CKPT}")

    import copy

    def run_epochs(epochs, lr, best_qwk, history, best_state_holder):
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        patience_counter = 0
        for epoch in range(epochs):
            global_epoch = len(history)  # continuous across freeze -> finetune, not reset per stage
            model.train()
            for batch in tqdm(train_loader, desc=f"[teacher] epoch {global_epoch}"):
                macula, disc, y = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE), batch["label"].to(DEVICE)
                out = model(macula, disc)
                loss = coral_loss(out["logit_dual"], y, pos_weight=pos_weight) + lambda_aux * aux_loss(out, y, pos_weight=pos_weight)
                opt.zero_grad(); loss.backward(); opt.step()

            val_qwk = quick_val_qwk(model, val_loader, DEVICE, "dual")
            history.append({"epoch": global_epoch, "val_qwk": val_qwk})
            print(f"epoch {global_epoch}: val_QWK={val_qwk:.4f}")
            if val_qwk > best_qwk:
                best_qwk, patience_counter = val_qwk, 0
                best_state_holder["state"] = copy.deepcopy(model.state_dict())
                robust_torch_save({"model_state": best_state_holder["state"], "epoch": global_epoch, "val_qwk": val_qwk}, out_ckpt)
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {global_epoch}")
                    break
        return best_qwk

    history = []
    best_state_holder = {"state": None}
    for p in model.backbone.parameters():
        p.requires_grad = False
    best_qwk = run_epochs(freeze_epochs, freeze_lr, -1.0, history, best_state_holder)

    # Reload the best freeze-phase weights from the IN-MEMORY copy, not from Drive. A real run hit
    # exactly this handoff: torch.save() had just written out_ckpt, and the very next line's
    # torch.load() of that same path raised FileNotFoundError -- Google Drive's FUSE mount can lag
    # a beat behind its own writes. The disk copy (saved above) still exists for persistence across
    # sessions; this in-memory path just removes the race for the handoff that happens within the
    # same function call.
    model.load_state_dict(best_state_holder["state"])
    for p in model.backbone.parameters():
        p.requires_grad = True
    best_qwk = run_epochs(finetune_epochs, 1e-5, best_qwk, history, best_state_holder)

    pd.DataFrame(history).to_csv(f"{LOGS_DIR}/teacher_history.csv", index=False)
    print(f"Teacher training done. Best val QWK = {best_qwk:.4f}")
    return out_ckpt

TEACHER_CKPT = train_teacher()

# Gate 2 check
_teacher = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
_teacher.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
_val_ds = DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform)
_val_loader = DataLoader(_val_ds, batch_size=16, shuffle=False, num_workers=2)
_gain = compute_dual_view_gain(_teacher, _val_loader, DEVICE)
print("Gate 2 check:", _gain)
GATE2_PASSED = _gain["QWK_dual"] > max(_gain["QWK_macula"], _gain["QWK_disc"])
if not GATE2_PASSED:
    print("*** GATE 2 WARNING: teacher dual-view does NOT beat its own auxiliary heads. "
          "Do not proceed to CSD training without investigating. ***")
    print(f"*** train_teacher() defaults are already lambda_aux=0.3, freeze_lr=3e-4 (lowered from "
          f"0.5/1e-3 after repeated real runs landed right on this margin -- see "
          f"experiment/documentation.md Section 0.2). If it STILL fails after several retrains "
          f"(train_teacher(force=True)) at these defaults, this is a real, reportable signal, not "
          f"noise -- try a different seed (train_teacher(force=True, seed=123)) as one more check, "
          f"then consider documenting a negative Gate 2 result honestly rather than retrying "
          f"indefinitely; RQ1 can still be answered (possibly as 'no' with analysis) per "
          f"docs/roadmap.md's explicit permission to report negative results. "
          f"GATE2_PASSED is a global flag -- later CSD cells check it and warn loudly rather than "
          f"silently training on top of an unresolved Gate 2. ***")
else:
    print("Gate 2: PASSED.")
del _teacher, _val_loader

## 12. Generic student-condition trainer

One function drives every student condition (baselines, no-distill, standard KD, CSD, and the
counterfactual-CSD ablation) so the training logic can't drift between conditions the way it
did across ad-hoc scripts in the original v1 design (technical doc Critical Issue 1).

In [ ]:
_teacher_cache = None
def get_teacher():
    global _teacher_cache
    if _teacher_cache is None:
        _teacher_cache = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        _teacher_cache.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
        _teacher_cache.eval()
        for p in _teacher_cache.parameters():
            p.requires_grad = False
    return _teacher_cache

def train_student_condition(run_name, seed, view_mode, alpha=0.0, beta=0.0, lambda_aux=0.5,
                             csd_variant="smoothl1_norm", tau_kd=2.0, tau_csd=0.5,
                             use_counterfactual_csd=False, epochs=40, patience=8, lr=1e-3,
                             batch_size=16, gamma_feat=0.0, huber_beta=1.0,
                             weight_decay=1e-4, force=False):
    """REV3 training-loop changes:
      * AdamW + cosine-annealed LR instead of flat Adam. v2's validation QWK oscillated hard
        (e.g. 0.51 -> 0.35 -> 0.46 between consecutive epochs), which both wasted the patience
        budget and made "best epoch" partly a lottery. A decaying LR settles the late epochs.
      * epochs 30 -> 40 and patience 5 -> 8, since the larger student needs longer to converge
        and the noisier early epochs were tripping early-stop prematurely.
      * Per-epoch gradient-norm logging for every loss component (judge.md Flag 6), measured on
        one batch so the cost is negligible. This is what makes "is CSD actually contributing?"
        answerable DURING the run rather than by hand-reading logs afterwards.
      * gamma_feat / feature-KD support for the new feature-KD control baseline.
    """
    ckpt_dir = f"{CKPT_DIR}/student/{run_name}"
    os.makedirs(ckpt_dir, exist_ok=True)
    out_ckpt = f"{ckpt_dir}/best_seed{seed}.pt"
    _probe_student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out_ckpt, _probe_student, unwrap_key="model_state"):
        print(f"{out_ckpt} already exists and matches this architecture, skipping (set force=True to redo).")
        return out_ckpt, None
    del _probe_student

    if beta > 0 and not GATE2_PASSED:
        print(f"*** [{run_name}|seed{seed}] proceeding despite Gate 2 not having passed for the "
              f"teacher -- CSD's Delta signal is only meaningful if the teacher itself shows a "
              f"real dual-view advantage. Results from this run are still saved and reported, "
              f"but treat them as provisional until Gate 2 is resolved. ***")

    set_seed(seed)
    pos_weight = compute_pos_weights(DRTID_TRAIN_CSV).to(DEVICE)
    train_ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, train_transform)
    val_ds = DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform)
    g = make_generator(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    teacher = get_teacher()
    student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student.backbone.load_state_dict(robust_torch_load(LIGHTWEIGHT_BACKBONE_CKPT, map_location=DEVICE))

    # Feature-KD needs a learned projection from teacher fused-dim to student fused-dim.
    feat_projector = None
    trainable = list(student.parameters())
    if gamma_feat > 0:
        feat_projector = nn.Linear(teacher.fusion.out_dim, student.fusion.out_dim).to(DEVICE)
        trainable = trainable + list(feat_projector.parameters())

    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)
    grad_probe_params = list(student.fusion.parameters()) + list(student.main_head.parameters())
    best_qwk, patience_counter, history = -1.0, 0, []

    for epoch in range(epochs):
        student.train()
        epoch_logs, epoch_gnorms = [], {}
        for bi, batch in enumerate(tqdm(train_loader, desc=f"[{run_name}|seed{seed}] epoch {epoch}")):
            macula, disc, y = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE), batch["label"].to(DEVICE)
            with torch.no_grad():
                teacher_out = teacher(macula, disc)
                teacher_cf_out = teacher.counterfactual_forward(macula, disc) if use_counterfactual_csd else None
            student_out = get_student_output(student, macula, disc, view_mode)
            student_cf_out = student.counterfactual_forward(macula, disc) if (use_counterfactual_csd and view_mode == "dual") else None

            loss, log, components = combined_student_loss(
                teacher_out, student_out, y, view_mode, alpha=alpha, beta=beta, lambda_aux=lambda_aux,
                tau_kd=tau_kd, csd_variant=csd_variant, tau_csd=tau_csd, pos_weight=pos_weight,
                use_counterfactual_csd=use_counterfactual_csd, teacher_cf_out=teacher_cf_out,
                student_cf_out=student_cf_out, gamma_feat=gamma_feat, feat_projector=feat_projector,
                huber_beta=huber_beta,
            )

            # Measure per-component gradient norms once per epoch (first batch only).
            if bi == 0 and view_mode == "dual":
                epoch_gnorms = component_grad_norms(components, grad_probe_params)
                student.zero_grad(set_to_none=True)

            opt.zero_grad(); loss.backward(); opt.step()
            epoch_logs.append(log)
        sched.step()

        val_qwk = quick_val_qwk(student, val_loader, DEVICE, view_mode)
        mean_log = {k: float(np.mean([l[k] for l in epoch_logs if k in l])) for k in epoch_logs[0]}
        mean_log.update({"epoch": epoch, "val_qwk": val_qwk, "lr": sched.get_last_lr()[0]})
        mean_log.update(epoch_gnorms)
        history.append(mean_log)
        loss_summary = {k: round(v, 4) for k, v in mean_log.items() if k.startswith("L_")}
        gnorm_summary = {k.replace("gnorm_", ""): round(v, 4) for k, v in epoch_gnorms.items()}
        msg = f"epoch {epoch}: val_QWK={val_qwk:.4f}  losses={loss_summary}"
        if gnorm_summary:
            msg += f"  gradnorms={gnorm_summary}"
        print(msg)

        if val_qwk > best_qwk:
            best_qwk, patience_counter = val_qwk, 0
            robust_torch_save({"model_state": student.state_dict(), "epoch": epoch, "val_qwk": val_qwk, "seed": seed}, out_ckpt)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    pd.DataFrame(history).to_csv(f"{LOGS_DIR}/{run_name}_seed{seed}_history.csv", index=False)
    print(f"[{run_name}|seed{seed}] best val QWK = {best_qwk:.4f} -> {out_ckpt}")
    return out_ckpt, history

print("train_student_condition defined.")

## 13. Day 5–6 — Baselines, no-distill, standard KD

Single-view baselines use 1 seed (not part of the core 3-seed RQ1 comparison). The three core
conditions (`dual_no_distill`, `dual_logitkd`, `dual_csd`) use 3 seeds each, mean±std reported
— a single seed is not enough to claim CSD is better on a dataset this size (technical doc
Critical Issue 8).

In [ ]:
train_student_condition("macula_only", seed=42, view_mode="macula_only", alpha=0.0, beta=0.0)
train_student_condition("disc_only", seed=42, view_mode="disc_only", alpha=0.0, beta=0.0)

In [ ]:
for seed in SEEDS:
    train_student_condition("dual_no_distill", seed=seed, view_mode="dual", alpha=0.0, beta=0.0, lambda_aux=0.5)

In [ ]:
for seed in SEEDS:
    train_student_condition("dual_logitkd", seed=seed, view_mode="dual", alpha=0.5, beta=0.0, lambda_aux=0.5, tau_kd=2.0)

In [ ]:
# REV3 ADDITION -- feature-KD control (judge.md Section G item 3).
# This is the control that makes CSD's central claim falsifiable. dual_logitkd only controls for
# "does ANY distillation help"; feature-KD controls for "is there anything special about
# distilling the DECISION-shift specifically, versus ordinary representation transfer?". If CSD
# does not beat feature-KD, the novelty claim has to be scoped down accordingly -- and saying so
# honestly is far stronger than not having run the control at all.
for seed in SEEDS:
    train_student_condition("dual_featkd", seed=seed, view_mode="dual", alpha=0.5, beta=0.0,
                             lambda_aux=0.5, tau_kd=2.0, gamma_feat=1.0)

## 14. Day 7 — Gate 3, CSD grid search, final DR-VERGE training

Gate 3 checks the teacher actually shows a non-trivial complementarity signal on validation
data *before* spending a full grid search + 3-seed training on it. Grid search is run on 1
seed only, fixed search space (no combinations added after seeing results — judge.md Flag 13
overfitting-to-Set-B risk), then the winning config is trained with the full 3-seed protocol.

In [ ]:
@torch.no_grad()
def gate3_check(teacher, val_loader, device):
    teacher.eval()
    batch = next(iter(val_loader))
    macula, disc = batch["macula"].to(device), batch["disc"].to(device)
    out = teacher(macula, disc)
    delta_t = _compute_delta(out["p_dual"], out["p_macula"], out["p_disc"])
    l1_norm = delta_t.abs().sum(dim=1).mean().item()
    frac_nontrivial = (delta_t.abs().sum(dim=1) > 0.02).float().mean().item()
    print(f"Gate 3: mean L1(Delta^T)={l1_norm:.4f}, fraction of samples with |Delta^T|>0.02: {frac_nontrivial:.2%}")
    if l1_norm < 1e-3:
        print("*** GATE 3 WARNING: teacher shift is near zero -- CSD has little signal to distill. "
              "Revisit Gate 2 before proceeding. ***")
    else:
        print("Gate 3: PASSED (non-trivial complementarity signal present).")
    return l1_norm, frac_nontrivial

if not GATE2_PASSED:
    print("*** Reminder: Gate 2 did not pass for this teacher. Gate 3's signal check below is "
          "still worth running (it may itself explain WHY Gate 2 failed -- e.g. a near-zero "
          "Delta would mean the teacher never learned to use both views jointly), but don't "
          "commit to the CSD grid search below until Gate 2 is understood. ***")

_val_loader = DataLoader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size=16, shuffle=True, num_workers=2)
gate3_check(get_teacher(), _val_loader, DEVICE)

In [ ]:
# Grid search: 1 seed (42), fixed search space, Set B (val) selection only -- decided BEFORE
# looking at results, per docs/roadmap.md Day 7 tie-break rule (QWK -> severe error -> simplicity).
#
# REV3 CHANGE -- the search space now spans a meaningful RANGE of CSD strength.
# v2's grid only varied beta over {0.5, 0.7}. Because the un-normalized Delta made L_CSD ~0.014,
# every point in that grid produced a CSD contribution of well under 1% of total loss -- so the
# "grid search" was really comparing four indistinguishable runs and could never have found a
# setting where CSD mattered. With the scale-normalized variant, beta is now a meaningful knob and
# the grid covers weak / moderate / strong / dominant CSD, plus the two v2 formulations kept as
# controlled ablations so the before/after comparison is explicit rather than implied.
# Beta values chosen from a MEASURED gradient ratio, not guessed. With the scale-normalized
# variant, beta=1.0 puts the CSD gradient at roughly 5x the task gradient at initialization
# (measured on a real DRTiD batch during rev3 validation) -- i.e. the opposite failure mode from
# v2, where CSD was ~0. The grid therefore brackets the balanced region from clearly-subordinate
# (0.1, ratio ~0.5x) through roughly-equal (0.2) up to CSD-leaning (0.5), and lets Set B pick.
GRID = [
    {"csd_variant": "smoothl1_norm", "alpha": 0.5, "beta": 0.1},
    {"csd_variant": "smoothl1_norm", "alpha": 0.5, "beta": 0.2},
    {"csd_variant": "smoothl1_norm", "alpha": 0.5, "beta": 0.5},
    {"csd_variant": "smoothl1_norm", "alpha": 0.25, "beta": 0.2},   # weaker logit-KD, CSD relatively stronger
    {"csd_variant": "magnitude_weighted_direction", "alpha": 0.5, "beta": 0.2},  # judge.md Flag 5
    {"csd_variant": "smoothl1", "alpha": 0.5, "beta": 0.7},         # v2 formulation, ABLATION (expected weak)
    {"csd_variant": "kl_softmax", "alpha": 0.5, "beta": 0.5},       # v1 formulation, ABLATION (L_CSD collapsed to 0 in v2)
]

grid_results = []
for combo in GRID:
    run_name = "grid_{}_a{}_b{}".format(combo["csd_variant"], combo["alpha"], combo["beta"])
    ckpt, history = train_student_condition(run_name, seed=42, view_mode="dual", lambda_aux=0.5, **combo)
    if history is None:  # already-trained checkpoint found, reload its recorded best QWK
        state = robust_torch_load(ckpt, map_location=DEVICE)
        best_qwk = state["val_qwk"]
    else:
        best_qwk = max(h["val_qwk"] for h in history)

    student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student.load_state_dict(robust_torch_load(ckpt, map_location=DEVICE)["model_state"])
    y_true, y_pred, _ = get_predictions(student, _val_loader, DEVICE, "dual")
    metrics = compute_metrics(y_true, y_pred)
    grid_results.append({**combo, "run_name": run_name, "val_QWK": best_qwk, "SevereErrorRate": metrics["SevereErrorRate"]})
    del student

grid_df = pd.DataFrame(grid_results).sort_values(["val_QWK", "SevereErrorRate"], ascending=[False, True])
grid_df.to_csv(f"{METRICS_DIR}/csd_grid_search.csv", index=False)
print(grid_df)

BEST_CSD_VARIANT = grid_df.iloc[0]["csd_variant"]
BEST_ALPHA = grid_df.iloc[0]["alpha"]
BEST_BETA = grid_df.iloc[0]["beta"]
print(f"\nBest CSD config: variant={BEST_CSD_VARIANT}, alpha={BEST_ALPHA}, beta={BEST_BETA}")

In [ ]:
# Final DR-VERGE training: 3 seeds at the winning grid config.
for seed in SEEDS:
    train_student_condition("dual_csd", seed=seed, view_mode="dual", alpha=BEST_ALPHA, beta=BEST_BETA,
                             lambda_aux=0.5, csd_variant=BEST_CSD_VARIANT, tau_kd=2.0, tau_csd=0.5)

In [ ]:
# Bonus ablation (1 seed): same-head counterfactual CSD -- judge.md's most important suggested
# check. Confirms whether the default head-based Delta's apparent complementarity gain survives
# once head-discrepancy is removed as a possible confound.
train_student_condition("dual_csd_counterfactual", seed=42, view_mode="dual", alpha=BEST_ALPHA, beta=BEST_BETA,
                         lambda_aux=0.5, csd_variant=BEST_CSD_VARIANT, use_counterfactual_csd=True)

## 15. Day 8 — PTQ INT8 quantization

Applied to the best `dual_csd` seed (highest val QWK among the 3). Quantizes **only the CNN
backbone**, not the whole student model -- confirmed by direct local testing that wrapping the
entire model between one QuantStub/DeQuantStub pair fails at runtime as soon as a quantized
tensor reaches the fusion block (`aten::native_batch_norm` has no QuantizedCPU kernel when fusion
used `BatchNorm1d`, and the same class of failure applies to `InteractionFusion`'s LayerNorm +
Linear + `torch.cat`/elementwise-product ops too), since none of `torch.cat`, normalization
layers, or `CORALHead`'s cumsum/softplus/sigmoid math have quantized-kernel equivalents in eager
mode by default. Quantizing just the backbone (the actual compute/parameter-heavy part) avoids
this while still capturing the real efficiency win; the fusion + both CORALHeads stay FP32.

Gate 5 verifies the model is *actually* INT8 by checking each module's **full path** for
"quantized" (e.g. `torch.ao.nn.intrinsic.quantized.modules.conv_relu.ConvReLU2d`) -- not just
the short class name, which for PyTorch's fused quantized modules doesn't contain the word
"Quantized" at all (confirmed directly: a correctly-converted model's fused conv module has
class name `ConvReLU2d`, easy to miss with a naive name-only check). If eager mode fails
outright, this cell reports it clearly rather than silently producing a broken model — treat
that as a signal to fall back to FX-graph-mode quantization (technical doc Section 10.2) or
mark RQ2 as future work, per the roadmap's explicit permission to report PTQ as not-achieved if
it doesn't work out cleanly.

In [ ]:
from torch.ao.quantization import prepare, convert, get_default_qconfig, QuantStub, DeQuantStub

class QuantizableBackbone(nn.Module):
    # Wraps ONLY the CNN backbone, not the whole student. An earlier version of this cell
    # wrapped the entire DualViewLightStudent between a single QuantStub/DeQuantStub pair --
    # confirmed by direct local testing to fail at runtime with "Could not run
    # 'aten::native_batch_norm' with arguments from the 'QuantizedCPU' backend" the moment a
    # quantized tensor reached the fusion block. InteractionFusion's LayerNorm/Linear/elementwise
    # ops and CORALHead's cumsum/softplus/sigmoid math have no quantized-kernel equivalents in
    # eager mode either. Quantizing only the backbone -- the actual compute/parameter-heavy part
    # -- sidesteps all of this while still capturing the real efficiency win; fusion + both
    # CORALHeads stay FP32 (cheap regardless).
    def __init__(self, backbone):
        super().__init__()
        self.quant = QuantStub()
        self.backbone = backbone
        self.dequant = DeQuantStub()

    def forward(self, x):
        return self.dequant(self.backbone(self.quant(x)))

def pick_best_csd_seed():
    best_seed, best_qwk = None, -1.0
    for seed in SEEDS:
        ckpt = f"{CKPT_DIR}/student/dual_csd/best_seed{seed}.pt"
        if os.path.exists(ckpt):
            qwk = robust_torch_load(ckpt, map_location="cpu")["val_qwk"]
            if qwk > best_qwk:
                best_seed, best_qwk = seed, qwk
    return best_seed, best_qwk

BEST_CSD_SEED, BEST_CSD_QWK = pick_best_csd_seed()
print(f"Best dual_csd seed for PTQ: seed={BEST_CSD_SEED}, val_QWK={BEST_CSD_QWK:.4f}")

def run_ptq():
    torch.backends.quantized.engine = "fbgemm"  # x86 (Colab); use "qnnpack" for ARM/mobile deployment target
    student_fp32 = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    student_fp32.load_state_dict(robust_torch_load(f"{CKPT_DIR}/student/dual_csd/best_seed{BEST_CSD_SEED}.pt", map_location="cpu")["model_state"])
    student_fp32 = student_fp32.to("cpu").eval()
    student_fp32.backbone.fuse_model()  # fuse Conv-BN-ReLU trios inside the backbone only

    student_fp32.backbone = QuantizableBackbone(student_fp32.backbone)
    student_fp32.backbone.qconfig = get_default_qconfig("fbgemm")  # only this submodule has a qconfig -> only it gets quantized
    prepared = prepare(student_fp32, inplace=False)

    calib_ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform)
    calib_loader = DataLoader(calib_ds, batch_size=8, shuffle=True, num_workers=0)  # only ~50 small batches, not worth multiprocessing overhead
    with torch.no_grad():
        for i, batch in enumerate(calib_loader):
            prepared(batch["macula"], batch["disc"])
            if i >= 50:
                break

    quantized = convert(prepared, inplace=False)
    return quantized

def _is_quantized_module_tree(model):
    # PyTorch's fused quantized modules (e.g. ConvReLU2d) live under
    # torch.ao.nn.intrinsic.quantized.* -- "quantized" is in the MODULE PATH, not always in
    # the short class __name__, so checking __name__ alone (as an earlier version of this
    # check did) can silently report "not quantized" on a model that quantized correctly.
    # Confirmed directly: a real converted model showed class name "ConvReLU2d" with module
    # path "torch.ao.nn.intrinsic.quantized.modules.conv_relu" -- caught by this check, missed
    # by a name-only check.
    return any("quantized" in f"{type(m).__module__}".lower() for m in model.modules())

try:
    QUANTIZED_STUDENT = run_ptq()
    PTQ_SUCCEEDED = True
    has_quantized = _is_quantized_module_tree(QUANTIZED_STUDENT)
    print("Gate 5 check -- quantized module(s) present in backbone:", has_quantized)
    if not has_quantized:
        print("*** GATE 5 WARNING: no quantized modules found -- PTQ did not actually convert the model. ***")
        PTQ_SUCCEEDED = False
    else:
        # INT8_MODEL_PATH is the single source of truth for where the deployable INT8 artifact
        # ended up -- later cells (Section 16's evaluation row) read THIS variable rather than
        # re-guessing a filename, so a TorchScript-vs-fallback branch here can never silently
        # desync from what evaluation reports as the model size.
        try:
            scripted = torch.jit.script(QUANTIZED_STUDENT)
            INT8_MODEL_PATH = f"{CKPT_DIR}/student/dual_csd/int8_seed{BEST_CSD_SEED}.pt"
            torch.jit.save(scripted, INT8_MODEL_PATH)
            print(f"Gate 5: PASSED. INT8 TorchScript model saved to {INT8_MODEL_PATH} ({model_size_mb(INT8_MODEL_PATH):.2f} MB)")
        except Exception as script_err:
            # Fallback if TorchScript export hits an environment-specific snag: state_dict
            # size is a reasonable proxy (INT8 weights are still packed int8, not FP32) even
            # though TorchScript is the more rigorous deployment-artifact comparison
            # (judge.md Code issue 5).
            print(f"torch.jit.script failed ({script_err!r}), falling back to state_dict size.")
            INT8_MODEL_PATH = f"{CKPT_DIR}/student/dual_csd/int8_seed{BEST_CSD_SEED}_statedict.pt"
            robust_torch_save(QUANTIZED_STUDENT.state_dict(), INT8_MODEL_PATH)
            print(f"Gate 5: PASSED (quantized modules confirmed). Saved state_dict fallback to {INT8_MODEL_PATH} ({model_size_mb(INT8_MODEL_PATH):.2f} MB)")
except Exception as e:
    print(f"*** PTQ FAILED: {e!r} ***")
    print("Per docs/roadmap.md: report RQ2/PTQ as future work rather than debugging further under time pressure.")
    PTQ_SUCCEEDED = False

## 16. Day 8 — Full evaluation across all conditions (Set C / official test, touched once)

In [ ]:
TEST_LOADER = DataLoader(DRTiDDualViewDataset(DRTID_TEST_CSV, eval_transform), batch_size=16, shuffle=False, num_workers=2)
_sample_batch = next(iter(TEST_LOADER))

FIELDNAMES = ["condition", "seed", "QWK", "MAE", "SevereErrorRate", "MacroF1",
              "Sensitivity_Grade0", "Sensitivity_Grade1", "Sensitivity_Grade2", "Sensitivity_Grade3", "Sensitivity_Grade4",
              "OrdinalViolationRate", "QWK_dual", "QWK_macula", "QWK_disc",
              "DualViewGain_G_internal", "DualViewGain_G_external",
              "ShiftMAE", "CosAgree", "BenefitCorr", "Brier", "ECE",
              "ModelSize_MB", "ParamCount", "CPU_Latency_median_ms", "CPU_Latency_p95_ms"]

# External-gain reference points: the INDEPENDENTLY trained single-view students (judge.md Flag 8).
# Filled in as those two rows are evaluated, then used for every dual-view condition.
INDEP_SINGLE_VIEW_QWK = {}

def blank_row():
    return {k: np.nan for k in FIELDNAMES}

def evaluate_one(model, view_mode, condition, seed, ckpt_path, teacher_for_shift=None):
    row = blank_row()
    row["condition"], row["seed"] = condition, seed
    y_true, y_pred, p_cum = get_predictions(model, TEST_LOADER, DEVICE, view_mode)
    row.update(compute_metrics(y_true, y_pred, p_cum))
    row.update(compute_calibration(p_cum, y_true))          # REV3: judge.md Flag 4
    if view_mode == "dual":
        row.update(compute_dual_view_gain(model, TEST_LOADER, DEVICE))
        # REV3: external gain vs the independently trained single-view students (judge.md Flag 8)
        if INDEP_SINGLE_VIEW_QWK.get("macula_only") is not None and INDEP_SINGLE_VIEW_QWK.get("disc_only") is not None:
            row["DualViewGain_G_external"] = compute_external_gain(
                row["QWK_dual"], INDEP_SINGLE_VIEW_QWK["macula_only"], INDEP_SINGLE_VIEW_QWK["disc_only"])
        # REV3: shift-fidelity -- did CSD actually transfer the pattern? (judge.md Flag 10)
        if teacher_for_shift is not None:
            row.update(compute_shift_fidelity(teacher_for_shift, model, TEST_LOADER, DEVICE))
    row["ModelSize_MB"] = model_size_mb(ckpt_path) if os.path.exists(ckpt_path) else np.nan
    row["ParamCount"] = param_count(model)
    macula, disc = _sample_batch["macula"].to(DEVICE), _sample_batch["disc"].to(DEVICE)
    lat = measure_cpu_latency(model, macula, disc, view_mode)
    row.update(lat)
    return row

rows = []

# Teacher
teacher = get_teacher()
rows.append(evaluate_one(teacher, "dual", "teacher", "-", TEACHER_CKPT))

# Single-seed conditions
for condition, view_mode in [("macula_only", "macula_only"), ("disc_only", "disc_only")]:
    ckpt = f"{CKPT_DIR}/student/{condition}/best_seed42.pt"
    model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    model.load_state_dict(robust_torch_load(ckpt, map_location=DEVICE)["model_state"])
    _row = evaluate_one(model, view_mode, condition, 42, ckpt)
    INDEP_SINGLE_VIEW_QWK[condition] = _row["QWK"]   # reference for external dual-view gain
    rows.append(_row)
    del model

# Multi-seed core conditions
for condition in ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]:
    for seed in SEEDS:
        ckpt = f"{CKPT_DIR}/student/{condition}/best_seed{seed}.pt"
        if not os.path.exists(ckpt):
            continue
        model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        model.load_state_dict(robust_torch_load(ckpt, map_location=DEVICE)["model_state"])
        rows.append(evaluate_one(model, "dual", condition, seed, ckpt, teacher_for_shift=teacher))
        del model

# Counterfactual CSD ablation (1 seed)
_cf_ckpt = f"{CKPT_DIR}/student/dual_csd_counterfactual/best_seed42.pt"
if os.path.exists(_cf_ckpt):
    model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    model.load_state_dict(robust_torch_load(_cf_ckpt, map_location=DEVICE)["model_state"])
    rows.append(evaluate_one(model, "dual", "dual_csd_counterfactual", 42, _cf_ckpt, teacher_for_shift=teacher))
    del model

# INT8 quantized best dual_csd (if PTQ succeeded)
if PTQ_SUCCEEDED:
    # REV3: this block now measures the quantized model's dual-view gain, not just its overall QWK.
    # RQ2 asks specifically whether PTQ PRESERVES the dual-view advantage or erodes it
    # disproportionately -- v2 recorded only QWK and left DualViewGain as NaN for the INT8 row,
    # so the actual RQ2 sub-question was never measured. Running all three view modes through the
    # quantized model closes that gap.
    QUANTIZED_STUDENT.eval()
    int8_preds = {}
    with torch.no_grad():
        for vm in ["dual", "macula_only", "disc_only"]:
            yt, yp, pc = [], [], []
            for batch in TEST_LOADER:
                m_cpu, d_cpu = batch["macula"].cpu(), batch["disc"].cpu()
                if vm == "dual":
                    out = QUANTIZED_STUDENT(m_cpu, d_cpu)
                    p = out["p_dual"]
                elif vm == "macula_only":
                    p = QUANTIZED_STUDENT.forward_single(m_cpu, which="macula")["p"]
                else:
                    p = QUANTIZED_STUDENT.forward_single(d_cpu, which="disc")["p"]
                yp.extend((p > 0.5).sum(dim=1).tolist())
                yt.extend(batch["label"].tolist())
                pc.append(p)
            int8_preds[vm] = (np.array(yt), np.array(yp), torch.cat(pc, dim=0))

    y_true, y_pred, p_cum_t = int8_preds["dual"]
    row = blank_row()
    row["condition"], row["seed"] = "dual_csd_int8_ptq", BEST_CSD_SEED
    row.update(compute_metrics(y_true, y_pred, p_cum_t))
    row.update(compute_calibration(p_cum_t, y_true))

    qwk_int8_dual = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    qwk_int8_mac = cohen_kappa_score(*int8_preds["macula_only"][:2], weights="quadratic")
    qwk_int8_disc = cohen_kappa_score(*int8_preds["disc_only"][:2], weights="quadratic")
    row["QWK_dual"], row["QWK_macula"], row["QWK_disc"] = qwk_int8_dual, qwk_int8_mac, qwk_int8_disc
    row["DualViewGain_G_internal"] = qwk_int8_dual - max(qwk_int8_mac, qwk_int8_disc)
    if INDEP_SINGLE_VIEW_QWK.get("macula_only") is not None and INDEP_SINGLE_VIEW_QWK.get("disc_only") is not None:
        row["DualViewGain_G_external"] = compute_external_gain(
            qwk_int8_dual, INDEP_SINGLE_VIEW_QWK["macula_only"], INDEP_SINGLE_VIEW_QWK["disc_only"])

    row["ModelSize_MB"] = model_size_mb(INT8_MODEL_PATH)  # set in Section 15's PTQ cell -- single source of truth
    torch.set_num_threads(1)
    times = []
    m_cpu, d_cpu = _sample_batch["macula"][:1].cpu(), _sample_batch["disc"][:1].cpu()
    with torch.no_grad():
        for _ in range(10):
            QUANTIZED_STUDENT(m_cpu, d_cpu)
        for _ in range(50):
            t0 = time.perf_counter(); QUANTIZED_STUDENT(m_cpu, d_cpu); times.append((time.perf_counter()-t0)*1000)
    row["CPU_Latency_median_ms"] = float(np.median(times))
    row["CPU_Latency_p95_ms"] = float(np.percentile(times, 95))
    rows.append(row)

raw_df = pd.DataFrame(rows, columns=FIELDNAMES)
raw_df.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)
print(f"Saved {len(raw_df)} rows to {METRICS_DIR}/all_conditions_raw.csv")
raw_df

## 17. Seed aggregation + clustered bootstrap CIs

`n=3` seeds is too small to claim statistical significance from mean±std alone (technical
doc Critical Issue 8 / judge.md Flag 11) — the bootstrap below resamples DRTiD's `ID` field
(clusters), not individual images independently, on the reasoning that two eyes from the same
patient are unlikely to be independent observations. As noted in Section 4: DRTiD's public
`ID` is a per-eye identifier with no separate patient field, so this clustering is a
best-effort safeguard given what the released metadata exposes, not a verified true-patient
grouping. Report it as "clustered by record ID" in the paper, not "clustered by patient",
unless DRTiD's own paper documents an eye-to-patient mapping this notebook doesn't have access
to.

In [ ]:
numeric_cols = ["QWK", "MAE", "SevereErrorRate", "MacroF1", "DualViewGain_G_internal",
                "DualViewGain_G_external", "ShiftMAE", "CosAgree", "BenefitCorr", "Brier", "ECE"]
agg = raw_df.groupby("condition")[numeric_cols].agg(["mean", "std"])
agg.to_csv(f"{METRICS_DIR}/all_conditions_aggregated.csv")
print(agg)

In [ ]:
def clustered_bootstrap_qwk_diff(condition_a, condition_b, seed_a, seed_b, n_boot=2000, rng_seed=0):
    # Loads BOTH models' predictions on the test set, resamples patient_id with replacement,
    # recomputes QWK for each resample, reports the 95% CI of QWK(a) - QWK(b).
    ckpt_a = f"{CKPT_DIR}/student/{condition_a}/best_seed{seed_a}.pt"
    ckpt_b = f"{CKPT_DIR}/student/{condition_b}/best_seed{seed_b}.pt"
    model_a = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    model_a.load_state_dict(robust_torch_load(ckpt_a, map_location=DEVICE)["model_state"])
    model_b = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    model_b.load_state_dict(robust_torch_load(ckpt_b, map_location=DEVICE)["model_state"])

    test_df = pd.read_csv(DRTID_TEST_CSV)
    y_true_a, y_pred_a, _ = get_predictions(model_a, TEST_LOADER, DEVICE, "dual")
    y_true_b, y_pred_b, _ = get_predictions(model_b, TEST_LOADER, DEVICE, "dual")
    patient_ids = test_df["patient_id"].values
    unique_patients = np.unique(patient_ids)

    rng = np.random.default_rng(rng_seed)
    diffs = []
    for _ in range(n_boot):
        sampled_patients = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([np.where(patient_ids == p)[0] for p in sampled_patients])
        qwk_a = cohen_kappa_score(y_true_a[idx], y_pred_a[idx], weights="quadratic")
        qwk_b = cohen_kappa_score(y_true_b[idx], y_pred_b[idx], weights="quadratic")
        diffs.append(qwk_a - qwk_b)
    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    del model_a, model_b
    return {"mean_diff": float(diffs.mean()), "ci_low": float(ci_low), "ci_high": float(ci_high),
            "excludes_zero": bool(ci_low > 0 or ci_high < 0)}

bootstrap_results = {}
for comparison_name, cond_b in [("dual_csd_vs_no_distill", "dual_no_distill"),
                                 ("dual_csd_vs_logitkd", "dual_logitkd"),
                                 ("dual_csd_vs_featkd", "dual_featkd")]:
    r = clustered_bootstrap_qwk_diff("dual_csd", cond_b, BEST_CSD_SEED, SEEDS[0])
    bootstrap_results[comparison_name] = r
    verdict = "CI excludes zero -- difference is credible" if r["excludes_zero"] else "CI INCLUDES zero -- NOT a credible difference, do not claim CSD is better here"
    print("{}: mean_diff={:.4f}, 95% CI=[{:.4f}, {:.4f}]  ({})".format(
        comparison_name, r["mean_diff"], r["ci_low"], r["ci_high"], verdict))

pd.DataFrame(bootstrap_results).T.to_csv(f"{METRICS_DIR}/bootstrap_qwk_diffs.csv")

## 18. Gate 4 — RQ1 summary

In [ ]:
csd_mean = agg.loc["dual_csd", ("QWK", "mean")]
no_distill_mean = agg.loc["dual_no_distill", ("QWK", "mean")]
logitkd_mean = agg.loc["dual_logitkd", ("QWK", "mean")]
csd_severe = agg.loc["dual_csd", ("SevereErrorRate", "mean")]
no_distill_severe = agg.loc["dual_no_distill", ("SevereErrorRate", "mean")]

print("=" * 70)
print("GATE 4 -- RQ1 ANSWER")
print("=" * 70)
print(f"dual_csd        mean QWK = {csd_mean:.4f}")
print(f"dual_no_distill mean QWK = {no_distill_mean:.4f}   (CSD beats it: {csd_mean > no_distill_mean})")
print(f"dual_logitkd    mean QWK = {logitkd_mean:.4f}   (CSD competitive/better: {csd_mean >= logitkd_mean})")
print(f"dual_csd severe error = {csd_severe:.4f} vs dual_no_distill = {no_distill_severe:.4f} "
      f"(CSD does not worsen severe error: {csd_severe <= no_distill_severe})")
for _cmp in ["dual_csd_vs_no_distill", "dual_csd_vs_logitkd", "dual_csd_vs_featkd"]:
    if _cmp in bootstrap_results:
        _r = bootstrap_results[_cmp]
        print(f"{_cmp}: diff={_r['mean_diff']:+.4f} 95% CI=[{_r['ci_low']:+.4f}, {_r['ci_high']:+.4f}] "
              f"credible={_r['excludes_zero']}")
print("=" * 70)

# REV3 ADDITION -- shift-fidelity verdict (judge.md Flag 10).
# QWK alone cannot say whether CSD did what it CLAIMS to do. These numbers answer the mechanism
# question separately from the performance question, so the paper can report the two independently
# (e.g. "CSD demonstrably transfers the shift pattern but that does not translate into higher QWK"
# is a precise, publishable finding; "CSD didn't win" alone is not).
print("\nSHIFT-FIDELITY (did CSD transfer the complementarity pattern?)")
for cond in ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd", "dual_csd_counterfactual"]:
    sub = raw_df[raw_df["condition"] == cond]
    if len(sub) and not sub["ShiftMAE"].isna().all():
        print(f"  {cond:26s} ShiftMAE={sub['ShiftMAE'].mean():.4f} (lower=closer to teacher)  "
              f"CosAgree={sub['CosAgree'].mean():+.4f}  BenefitCorr={sub['BenefitCorr'].mean():+.4f}")
print("  If dual_csd shows lower ShiftMAE / higher CosAgree than the non-CSD baselines, CSD")
print("  provably transferred the shift pattern -- independently of whether QWK improved.")
print("=" * 70)
print("Per docs/roadmap.md: if these criteria are NOT met, that is still a valid, reportable")
print("finding (negative result with analysis) -- NOT an experiment failure. Report honestly.")

## 19. Chart generation

All figures saved as PNG under `results/figures/` on Drive.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 130
CORE_ORDER = ["teacher", "macula_only", "disc_only", "dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd",
              "dual_csd_counterfactual", "dual_csd_int8_ptq"]
present_order = [c for c in CORE_ORDER if c in raw_df["condition"].unique()]

def condition_mean_std(col):
    means, stds = [], []
    for c in present_order:
        vals = raw_df.loc[raw_df["condition"] == c, col].dropna()
        means.append(vals.mean())
        stds.append(vals.std() if len(vals) > 1 else 0.0)
    return means, stds

def bar_with_error(col, title, ylabel, fname):
    means, stds = condition_mean_std(col)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(present_order, means, yerr=stds, capsize=4, color="#4C72B0")
    ax.set_title(title); ax.set_ylabel(ylabel)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/{fname}", bbox_inches="tight")
    plt.show()

bar_with_error("QWK", "Quadratic Weighted Kappa by condition (mean ± std)", "QWK", "bar_qwk_comparison.png")
bar_with_error("DualViewGain_G_internal", "Internal dual-view gain by condition", "G = QWK_dual - max(QWK_macula, QWK_disc)", "bar_dual_view_gain.png")
bar_with_error("SevereErrorRate", "Severe error rate |y-yhat|>=2 by condition (lower is better)", "Severe error rate", "bar_severe_error.png")

In [ ]:
# Per-grade sensitivity heatmap
sens_cols = [f"Sensitivity_Grade{g}" for g in range(5)]
sens_matrix = np.array([[raw_df.loc[raw_df["condition"] == c, col].mean() for col in sens_cols] for c in present_order])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sens_matrix, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels([f"Grade {g}" for g in range(5)])
ax.set_yticks(range(len(present_order))); ax.set_yticklabels(present_order)
for i in range(sens_matrix.shape[0]):
    for j in range(sens_matrix.shape[1]):
        if not np.isnan(sens_matrix[i, j]):
            ax.text(j, i, f"{sens_matrix[i,j]:.2f}", ha="center", va="center",
                    color="white" if sens_matrix[i, j] > 0.5 else "black", fontsize=8)
ax.set_title("Per-grade sensitivity (recall) by condition")
plt.colorbar(im, ax=ax, label="Sensitivity")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/sensitivity_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# Efficiency tradeoff: model size vs QWK
fig, ax = plt.subplots(figsize=(8, 6))
for c in present_order:
    sub = raw_df[raw_df["condition"] == c]
    x, y = sub["ModelSize_MB"].mean(), sub["QWK"].mean()
    marker = "*" if "int8" in c else ("D" if c == "teacher" else "o")
    size = 300 if "int8" in c or c == "teacher" else 120
    ax.scatter(x, y, s=size, marker=marker, label=c)
ax.set_xlabel("Model size (MB)"); ax.set_ylabel("QWK")
ax.set_title("Efficiency tradeoff: model size vs accuracy")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/efficiency_scatter.png", bbox_inches="tight")
plt.show()

# Latency vs QWK
fig, ax = plt.subplots(figsize=(8, 6))
for c in present_order:
    sub = raw_df[raw_df["condition"] == c]
    x, y = sub["CPU_Latency_median_ms"].mean(), sub["QWK"].mean()
    marker = "*" if "int8" in c else ("D" if c == "teacher" else "o")
    size = 300 if "int8" in c or c == "teacher" else 120
    ax.scatter(x, y, s=size, marker=marker, label=c)
ax.set_xlabel("CPU latency, median ms (batch=1, 1 thread)"); ax.set_ylabel("QWK")
ax.set_title("Efficiency tradeoff: CPU latency vs accuracy")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/latency_scatter.png", bbox_inches="tight")
plt.show()

In [ ]:
# Training curves: val QWK per epoch for teacher + core 3 conditions
fig, ax = plt.subplots(figsize=(9, 6))
curve_specs = [("teacher_history.csv", "teacher")]
for cond in ["dual_no_distill", "dual_logitkd", "dual_csd"]:
    curve_specs.append((f"{cond}_seed{SEEDS[0]}_history.csv", cond))

for fname, label in curve_specs:
    path = f"{LOGS_DIR}/{fname}"
    if os.path.exists(path):
        hist = pd.read_csv(path)
        ax.plot(hist["epoch"], hist["val_qwk"], marker="o", markersize=3, label=label)

ax.set_xlabel("Epoch"); ax.set_ylabel("Validation QWK")
ax.set_title("Training curves (val QWK per epoch, seed 42)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/training_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# Delta^T distribution (Gate 3 diagnostic, full val set this time not just one batch)
teacher = get_teacher()
val_loader_full = DataLoader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size=16, shuffle=False, num_workers=2)
all_l1 = []
with torch.no_grad():
    for batch in val_loader_full:
        macula, disc = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE)
        out = teacher(macula, disc)
        delta_t = _compute_delta(out["p_dual"], out["p_macula"], out["p_disc"])
        all_l1.extend(delta_t.abs().sum(dim=1).cpu().tolist())

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(all_l1, bins=30, color="#55A868")
ax.axvline(0.02, color="red", linestyle="--", label="0.02 threshold (Gate 3 reference)")
ax.set_xlabel("|Delta^T|_1 (teacher complementarity shift, L1 norm)")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of teacher Delta shift on full validation set (median={np.median(all_l1):.4f})")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/delta_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices: teacher vs best dual_csd seed
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (label, model_ref) in zip(axes, [("Teacher (upper bound)", teacher), ("Best dual_csd student", None)]):
    if model_ref is None:
        model_ref = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        model_ref.load_state_dict(robust_torch_load(f"{CKPT_DIR}/student/dual_csd/best_seed{BEST_CSD_SEED}.pt", map_location=DEVICE)["model_state"])
    y_true, y_pred, _ = get_predictions(model_ref, TEST_LOADER, DEVICE, "dual")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(label)
    ax.set_xlabel("Predicted grade"); ax.set_ylabel("True grade")
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    for i in range(5):
        for j in range(5):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/confusion_matrices.png", bbox_inches="tight")
plt.show()

In [ ]:
# Ablation grid heatmap (CSD variant x alpha x beta, Set B QWK)
fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"{r.csd_variant}\na={r.alpha}, b={r.beta}" for r in grid_df.itertuples()]
ax.bar(labels, grid_df["val_QWK"], color="#C44E52")
ax.set_ylabel("Set B (val) QWK")
ax.set_title("CSD grid search results (1 seed, fixed search space)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/ablation_grid.png", bbox_inches="tight")
plt.show()
print(f"\nAll figures saved to {FIGURES_DIR}")

## 20. Final results dashboard

In [ ]:
print("=" * 78)
print("DR-VERGE -- FINAL RESULTS DASHBOARD")
print("=" * 78)
print("\nPer-condition (mean +/- std where applicable):\n")
display_cols = ["QWK", "MAE", "SevereErrorRate", "MacroF1",
                "DualViewGain_G_internal", "DualViewGain_G_external", "ShiftMAE", "CosAgree", "BenefitCorr"]
summary_rows = []
for c in present_order:
    sub = raw_df[raw_df["condition"] == c]
    row = {"condition": c, "n_seeds": len(sub)}
    for col in display_cols:
        vals = sub[col].dropna()
        row[col] = f"{vals.mean():.4f} +/- {vals.std():.4f}" if len(vals) > 1 else (f"{vals.mean():.4f}" if len(vals) else "-")
    summary_rows.append(row)
summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))

summary_table.to_csv(f"{METRICS_DIR}/final_summary_table.csv", index=False)
with open(f"{METRICS_DIR}/final_summary_table.md", "w") as f:
    f.write(summary_table.to_markdown(index=False))

print(f"\nAll checkpoints:  {CKPT_DIR}")
print(f"All metrics CSVs: {METRICS_DIR}")
print(f"All figures:      {FIGURES_DIR}")
print(f"Training logs:    {LOGS_DIR}")
print("\nRemember: Gate 4's verdict on RQ1 (Section 18 above) is the headline result --")
print("re-read it before writing the paper's Results section, and use judge.md Section I's")
print("safe phrasing for every claim (\'operational proxy\', not \'proves complementarity\').")

## Done

This notebook trained and evaluated every condition in `docs/roadmap.md`'s experiment matrix,
generated all figures, and saved everything to Drive. Next steps live outside this notebook:

- Read Section 18's Gate 4 verdict and Section 17's bootstrap CIs before writing the paper's
  Results section — they are the actual answer to RQ1, not a formality to skip.
- Pull the limitations list from `docs/judge.md` Section H / the "safe sentences" in Section I
  directly into the paper's Limitations section.
- If `PTQ_SUCCEEDED` is False, report RQ2 as future work rather than debugging further —
  RQ1 being fully and credibly answered matters more than a rushed RQ2.